# Module 1 — Triton GEMM Compiler Pipeline Exploration

## 1. Project Overview

This project investigates the internal compilation pipeline of Triton using a tiled General Matrix Multiplication (GEMM) kernel as a practical case study.

The objective is to understand how high-level Triton GPU programs are progressively transformed into lower-level intermediate representations (IR) and eventually compiled into NVIDIA GPU instructions.

Unlike traditional GPU kernel optimization projects that primarily focus on runtime performance, this project emphasizes **compiler architecture, MLIR-based intermediate representations, GPU-specific lowering, and hardware-aware code generation**.

We will implement a tiled GEMM kernel, inspect the generated TTIR, TTGIR, LLVM IR, and PTX, and analyze how high-level tensor computations are mapped onto GPU execution resources.

### Development Environment

- **Platform:** Google Colab
- **GPU:** NVIDIA GPU
- **Programming Language:** Python / Triton
- **Compiler Infrastructure:** Triton, MLIR, LLVM
- **Target Backend:** NVIDIA PTX
- **Workload:** Tiled GEMM with FP16 inputs and FP32 accumulation

---

## 2. Project Objectives

The main objectives are:

1. Implement and validate a tiled GEMM kernel using Triton.
2. Understand Triton's JIT compilation and GPU execution model.
3. Extract and inspect TTIR, TTGIR, LLVM IR, and PTX.
4. Identify SSA values, MLIR operations, and compiler transformations.
5. Understand the differences between TTIR and TTGIR.
6. Analyze GPU-specific tensor layouts and thread-to-data mappings.
7. Investigate how `tl.dot` is lowered into GPU-specific matrix computation operations.
8. Examine the relationship between compiler lowering and NVIDIA GPU instruction generation.
9. Connect compiler decisions with GPU hardware characteristics and potential performance implications.

### Central Research Question

**How does the Triton compiler transform a high-level tiled GEMM kernel into hardware-specific GPU instructions, and what information is introduced or transformed at each compilation stage?**

---

## 3. GEMM Workload

We consider the matrix multiplication:

\[
C=A\times B
\]

where:

\[
A\in\mathbb{R}^{M\times K},\quad
B\in\mathbb{R}^{K\times N},\quad
C\in\mathbb{R}^{M\times N}
\]

The computation is implemented using block-level tiling.

Each Triton program computes a tile of the output matrix:

\[
C_{\mathrm{tile}}
=
\sum_{k}A_{\mathrm{tile},k}B_{k,\mathrm{tile}}
\]

### Initial Kernel Configuration

```python
BLOCK_M = 32
BLOCK_N = 64
BLOCK_K = 32
num_warps = 4
```

These parameters serve as the initial configuration and may be adjusted according to the available GPU architecture and compiler constraints.

### Key Triton Operations

- `tl.program_id`: Identify the current program instance.
- `tl.arange`: Generate logical tensor indices.
- `tl.load`: Load tiled matrix data.
- `tl.dot`: Perform tiled matrix multiplication and accumulation.
- `tl.store`: Write output tiles into global memory.

The GEMM kernel provides a realistic workload for investigating tensor layouts, warp-level computation, compiler lowering, and GPU instruction generation.

---

## 4. Triton Compiler Pipeline

The simplified compilation pipeline is:

```text
                 Triton Python GEMM
                         |
                         v
                  Triton AST / JIT
                         |
                         v
                   Triton IR (TTIR)
                         |
                         | High-Level Transformations
                         v
                Triton GPU IR (TTGIR)
                         |
                         | GPU-Specific Lowering
                         v
              Lower-Level MLIR / LLVM IR
                         |
                         | NVIDIA Backend
                         v
                        PTX
                         |
                         | PTX Assembly / JIT
                         v
                GPU Machine Code
                      (SASS)
                         |
                         v
                    GPU Execution
```

This diagram represents the conceptual compilation pipeline. The actual implementation may involve additional dialects, intermediate representations, and optimization passes.

### 4.1 Triton IR (TTIR)

TTIR represents GPU computation using high-level Triton operations and tensor abstractions.

**Analysis Focus:**

- SSA definitions and uses
- Tensor shapes and data types
- Program indexing and address calculations
- Memory load and store operations
- Dot-product operations
- High-level compiler transformations

### 4.2 Triton GPU IR (TTGIR)

TTGIR introduces GPU-specific execution layouts and hardware mapping information.

**Analysis Focus:**

- Blocked tensor layouts
- Thread and warp organization
- Tensor distribution across execution resources
- Dot operand layouts
- Layout conversion operations
- GPU-specific optimization and lowering

### 4.3 LLVM IR

LLVM IR represents lower-level operations after Triton's GPU-specific lowering stages.

**Analysis Focus:**

- LLVM SSA values
- Control flow and basic blocks
- Pointer and address calculations
- Memory operations
- NVVM intrinsics and target-specific operations
- Lowering of matrix computation instructions

### 4.4 PTX

PTX is NVIDIA's virtual instruction set architecture for GPU programs.

**Analysis Focus:**

- Thread and block indexing
- Global memory instructions
- Predicate-based execution
- Matrix multiplication instructions, where applicable
- Address calculations
- Architecture-specific code generation

Depending on the GPU architecture, kernel configuration, and compiler version, the generated PTX may contain instructions such as `mma.sync` or other supported matrix computation instructions.

---

## 5. Experimental Plan

### Experiment 1.1 — Environment Inspection

**Objective:** Establish a reproducible Triton compiler environment.

**Tasks:**

- Identify the NVIDIA GPU model and compute capability.
- Inspect CUDA and PyTorch versions.
- Verify the Triton installation.
- Identify available compiler tools.
- Record the environment configuration.

**Deliverable:** Environment configuration summary.

### Experiment 1.2 — Tiled GEMM Implementation

**Objective:** Implement and execute a tiled GEMM kernel.

**Tasks:**

- Implement GEMM using `@triton.jit`.
- Configure `BLOCK_M`, `BLOCK_N`, and `BLOCK_K`.
- Implement tiled memory loads with boundary masking.
- Perform matrix multiplication using `tl.dot`.
- Accumulate results in FP32.
- Write results to global memory.
- Validate correctness against `torch.matmul`.

**Deliverable:** A working, validated Triton GEMM kernel.

### Experiment 1.3 — Compiler IR Extraction

**Objective:** Extract intermediate representations generated during Triton compilation.

**Tasks:**

- Access compilation artifacts through the compiled kernel object.
- Extract TTIR.
- Extract TTGIR.
- Extract LLVM IR.
- Extract PTX.
- Save generated artifacts for detailed inspection.

**Deliverable:**

```text
artifacts/
├── gemm.ttir
├── gemm.ttgir
├── gemm.llir
└── gemm.ptx
```

### Experiment 1.4 — TTIR Analysis

**Objective:** Understand the high-level representation of tiled GEMM.

**Tasks:**

- Identify SSA values and their dependencies.
- Examine Triton-specific MLIR operations.
- Analyze program indexing and tensor shapes.
- Investigate tiled memory accesses.
- Identify the representation of matrix multiplication.
- Understand how loops and accumulators are represented.

**Deliverable:** Annotated TTIR analysis.

### Experiment 1.5 — TTGIR Analysis

**Objective:** Understand how the compiler introduces GPU-specific execution information.

**Tasks:**

- Identify tensor layout attributes.
- Examine thread and warp mappings.
- Analyze blocked layouts and dot operand layouts.
- Identify layout conversion operations.
- Understand how tensor operations are mapped onto GPU execution resources.
- Compare selected operations between TTIR and TTGIR.

**Deliverable:** TTIR-to-TTGIR transformation analysis.

### Experiment 1.6 — LLVM IR & PTX Analysis

**Objective:** Analyze low-level lowering and GPU instruction generation.

**Tasks:**

- Identify generated LLVM functions and basic blocks.
- Examine memory-related operations and address calculations.
- Identify target-specific intrinsics where present.
- Inspect PTX load/store instructions.
- Identify GPU matrix computation instructions.
- Trace key GEMM operations from Triton source code to generated instructions.

**Deliverable:** LLVM IR and PTX analysis.

### Experiment 1.7 — Compiler Pipeline Summary

**Objective:** Consolidate the findings into a coherent understanding of the Triton compilation pipeline.

**Tasks:**

- Explain the purpose of each IR level.
- Compare high-level and GPU-specific representations.
- Summarize important lowering transformations.
- Explain how GEMM tiling and layouts relate to GPU execution.
- Document architecture-dependent observations.
- Identify potential compiler optimization opportunities.

**Deliverable:** Compiler pipeline technical summary.

---

## 6. Expected Repository Structure

```text
triton-compiler-optimization/
│
├── README.md
│
├── notebooks/
│   └── triton_gemm_compiler_pipeline.ipynb
│
├── artifacts/
│   ├── gemm.ttir
│   ├── gemm.ttgir
│   ├── gemm.llir
│   └── gemm.ptx
│
└── docs/
    └── compiler_pipeline_analysis.md
```

---

## 7. Key Technical Questions

After completing this notebook, we should be able to answer the following questions.

### Compiler Architecture

1. Why does Triton use multiple levels of intermediate representation?
2. What is the difference between TTIR and TTGIR?
3. What are MLIR dialects, and why are they useful?
4. How is SSA represented in Triton IR?
5. What is the difference between compiler optimization and lowering?

### GPU Execution Model

6. How does Triton map a logical program to GPU execution resources?
7. How are tensor elements distributed across threads and warps?
8. What role do GPU layouts play in code generation?
9. How does tiled GEMM improve data reuse?

### Compiler Lowering and Code Generation

10. How is `tl.dot` represented in compiler IR?
11. How are tensor operations lowered into target-specific instructions?
12. How does the compiler select matrix computation instructions?
13. What is the relationship between LLVM IR, PTX, and SASS?
14. Why might generated code differ across NVIDIA GPU architectures?

### Compiler Optimization

15. How can IR inspection help identify performance bottlenecks?
16. What are the potential costs of layout conversions?
17. How can tile sizes influence compiler-generated code?
18. Which compiler transformations could potentially improve GEMM performance?

---

## 8. Success Criteria

This module is complete when:

- [ ] The development environment is verified.
- [ ] A tiled GEMM kernel runs successfully on an NVIDIA GPU.
- [ ] The GEMM output is validated against PyTorch.
- [ ] TTIR, TTGIR, LLVM IR, and PTX are extracted.
- [ ] Important SSA values and compiler operations are identified.
- [ ] GPU-specific tensor layouts are analyzed.
- [ ] The representation of `tl.dot` is traced across relevant compilation stages.
- [ ] Key GPU instructions are identified in the generated PTX.
- [ ] A technical summary of the compiler pipeline is completed.

---

## 9. Expected Learning Outcomes

Upon completing this module, we should have a practical understanding of:

- Triton compiler architecture
- MLIR-based GPU compiler infrastructure
- SSA and intermediate representations
- GPU tensor layouts
- Thread and warp mapping
- Compiler optimization versus lowering
- LLVM and NVIDIA GPU code generation
- Hardware-aware compilation of tiled GEMM

The central outcome is the ability to **trace a nontrivial GPU computation through the compiler pipeline and explain how high-level tensor abstractions are transformed into hardware-specific operations**.

---

## 10. Connection to Future Work

This module provides the foundation for subsequent compiler optimization experiments.

### Next: Triton Kernel Fusion

Investigate fused and unfused GPU computations, intermediate memory traffic, and the impact of fusion on generated code.

### Next: GPU Performance Analysis

Evaluate the impact of tile configurations, warp counts, and compiler-generated instructions on GEMM performance.

### Long-Term: Custom MLIR Compiler Passes

Develop and evaluate compiler optimization passes using MLIR and C++.

**The ultimate goal is to move beyond writing efficient GPU kernels toward understanding and developing the compiler transformations that generate them.**

### Environment Inspection

In [1]:
# ============================================================
# Experiment 1.1 — Environment Inspection
# Project: Triton GEMM Compiler Pipeline Exploration
# ============================================================

import sys
import platform
import subprocess
import shutil
import importlib.util

# ------------------------------------------------------------
# 1. System Information
# ------------------------------------------------------------

print("=" * 60)
print("1. System Information")
print("=" * 60)

print("Python Version:", sys.version.split()[0])
print("Operating System:", platform.platform())


# ------------------------------------------------------------
# 2. NVIDIA GPU Information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("2. NVIDIA GPU Information")
print("=" * 60)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found.")


# ------------------------------------------------------------
# 3. PyTorch and CUDA
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("3. PyTorch and CUDA")
print("=" * 60)

import torch

print("PyTorch Version:", torch.__version__)
print("PyTorch CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    device = torch.device("cuda:0")
    props = torch.cuda.get_device_properties(device)

    print("GPU Name:", torch.cuda.get_device_name(device))
    print("Compute Capability:",
          torch.cuda.get_device_capability(device))

    print("Total GPU Memory (GB):",
          round(props.total_memory / (1024 ** 3), 2))

    print("SM Count:", props.multi_processor_count)

else:
    print("WARNING: CUDA GPU is unavailable.")


# ------------------------------------------------------------
# 4. Triton Compiler
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("4. Triton Compiler")
print("=" * 60)

if importlib.util.find_spec("triton") is not None:

    import triton
    import triton.language as tl

    print("Triton Version:", triton.__version__)
    print("Triton Location:", triton.__file__)

else:
    print("Triton is not installed.")


# ------------------------------------------------------------
# 5. Compiler Toolchain
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("5. Compiler Toolchain")
print("=" * 60)

compiler_tools = [
    "clang",
    "llvm-config",
    "mlir-opt",
    "ptxas",
    "nvcc",
]

for tool in compiler_tools:

    path = shutil.which(tool)

    print(f"{tool:15s}: {path or 'Not found in PATH'}")


# ------------------------------------------------------------
# 6. Environment Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("6. Environment Summary")
print("=" * 60)

print("GPU Ready:", torch.cuda.is_available())

print(
    "Triton Installed:",
    importlib.util.find_spec("triton") is not None
)

print("\nEnvironment inspection completed.")

1. System Information
Python Version: 3.13.16
Operating System: Linux-6.6.122+-x86_64-with-glibc2.39

2. NVIDIA GPU Information

3. PyTorch and CUDA
PyTorch Version: 2.11.0+cu130
PyTorch CUDA Version: 13.0
CUDA Available: True
GPU Name: Tesla T4
Compute Capability: (7, 5)
Total GPU Memory (GB): 14.56
SM Count: 40

4. Triton Compiler
Triton Version: 3.6.0
Triton Location: /usr/local/lib/python3.13/dist-packages/triton/__init__.py

5. Compiler Toolchain
clang          : Not found in PATH
llvm-config    : Not found in PATH
mlir-opt       : Not found in PATH
ptxas          : /usr/local/cuda/bin/ptxas
nvcc           : /usr/local/cuda/bin/nvcc

6. Environment Summary
GPU Ready: True
Triton Installed: True

Environment inspection completed.


In [2]:
import torch

a = torch.randn((512, 512), device="cuda", dtype=torch.float16)
b = torch.randn((512, 512), device="cuda", dtype=torch.float16)

c = torch.matmul(a, b)
torch.cuda.synchronize()

print("GPU:", torch.cuda.get_device_name(0))
print("Matrix multiplication:", c.shape)
print("Output dtype:", c.dtype)
print("CUDA execution: PASSED")

GPU: Tesla T4
Matrix multiplication: torch.Size([512, 512])
Output dtype: torch.float16
CUDA execution: PASSED


### Environment Summary

The development environment uses an NVIDIA Tesla T4 GPU with Compute Capability 7.5, PyTorch 2.11.0, CUDA 13.0, and Triton 3.6.0.

The NVIDIA CUDA compilation utilities, including `ptxas` and `nvcc`, are available. Standalone LLVM and MLIR utilities are not currently exposed through the system PATH.

No additional compiler installation is required for the initial Triton GEMM compilation and IR exploration experiments.

The project will use **FP16 GEMM with FP32 accumulation** to investigate GPU-specific compiler lowering and NVIDIA PTX code generation.

## Experiment 1.2 — Tiled GEMM Implementation

### 1. Objective

The objective of this experiment is to implement and execute a tiled General Matrix Multiplication (GEMM) kernel using Triton.

This experiment establishes a functional GPU workload for subsequent compiler pipeline exploration, including TTIR, TTGIR, LLVM IR, and PTX analysis.

The primary focus is to understand how high-level tensor computations are expressed in Triton, rather than achieving maximum GPU performance.

### 2. GEMM Computation

We implement the matrix multiplication:

$$
C = A \times B
$$

where:

$$
A \in \mathbb{R}^{M \times K}, \quad
B \in \mathbb{R}^{K \times N}, \quad
C \in \mathbb{R}^{M \times N}
$$

The computation uses block-level tiling, where each Triton program computes a tile of the output matrix.

### 3. Implementation Strategy

The kernel will perform the following operations:

1. Determine the output tile using `tl.program_id`.
2. Compute matrix indices using `tl.arange`.
3. Load input tiles from global memory using `tl.load`.
4. Perform matrix multiplication using `tl.dot`.
5. Accumulate intermediate results in FP32.
6. Store the output tile using `tl.store`.

Boundary masking will be used to handle matrix dimensions that are not multiples of the tile sizes.

### 4. Initial Kernel Configuration

The experiment will use the following initial configuration:

| Parameter | Value | Description |
|---|---|---|
| Input Data Type | FP16 | Matrix elements |
| Accumulator Type | FP32 | Intermediate accumulation |
| Output Data Type | FP16 | Output matrix |
| BLOCK_M | 32 | Output tile rows |
| BLOCK_N | 64 | Output tile columns |
| BLOCK_K | 32 | Reduction tile size |
| num_warps | 4 | Warps per Triton program |

These parameters may be adjusted based on the NVIDIA Tesla T4 architecture and compiler requirements.

### 5. Correctness Validation

The Triton GEMM output will be compared against `torch.matmul`.

We will verify:

- Output tensor shape and data type.
- Numerical correctness within an appropriate floating-point tolerance.
- Correct handling of boundary tiles.
- Successful GPU kernel compilation and execution.

### 6. Expected Outcomes

By completing this experiment, we should be able to:

- Understand the structure of a tiled Triton GEMM kernel.
- Explain the relationship between program IDs, tile sizes, and GPU execution.
- Understand the purpose of `tl.dot` and FP32 accumulation.
- Implement boundary-safe global memory operations.
- Launch and validate a Triton GPU kernel.
- Obtain a compiled kernel object for subsequent IR extraction.

### 7. Connection to Compiler Analysis

The compiled GEMM kernel will serve as the input for Experiment 1.3.

We will extract its intermediate representations and investigate how Triton transforms high-level operations such as `tl.load`, `tl.dot`, and `tl.store` into GPU-specific operations and instructions.

**Next: Experiment 1.3 — Compiler IR Extraction**

In [3]:
# M = 256
# N = 512
# K = 256

# BLOCK_M = 32
# BLOCK_N = 64
# BLOCK_K = 32

# num_warps = 4

In [4]:
# ============================================================
# Experiment 1.2 — Tiled GEMM Implementation
# ============================================================

import torch
import triton
import triton.language as tl

# Matrix dimensions
M = 256
N = 512
K = 256

# GEMM tile configuration
BLOCK_M = 32
BLOCK_N = 64
BLOCK_K = 32

NUM_WARPS = 4

# Random input matrices
torch.manual_seed(42)

A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)

C = torch.empty((M, N), device="cuda", dtype=torch.float16)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)

A shape: torch.Size([256, 256])
B shape: torch.Size([256, 512])
C shape: torch.Size([256, 512])


In [5]:
@triton.jit
def gemm_kernel(
    A_ptr,
    B_ptr,
    C_ptr,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    stride_am: tl.constexpr,
    stride_ak: tl.constexpr,
    stride_bk: tl.constexpr,
    stride_bn: tl.constexpr,
    stride_cm: tl.constexpr,
    stride_cn: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    # --------------------------------------------------------
    # Step 1 — Program ID
    # Get the program ID along M and N dimensions.
    # --------------------------------------------------------


    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)


    # --------------------------------------------------------
    # Step 2 — Generate Tile Indices
    # Generate row indices for A and C.
    # Generate column indices for B and C.
    # Generate reduction indices along K.
    # --------------------------------------------------------

    offsets_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offsets_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offsets_k = tl.arange(0, BLOCK_K)


    # --------------------------------------------------------
    # Step 3 — Initialize Accumulator
    # Initialize an FP32 accumulator with shape
    # (BLOCK_M, BLOCK_N).
    # --------------------------------------------------------


    accumulator = tl.full(
        (BLOCK_M, BLOCK_N),
        0,
        dtype=tl.float32,
    )



    # --------------------------------------------------------
    # Step 4 — GEMM Reduction Loop
    # --------------------------------------------------------

    for k_tile in range(tl.cdiv(K, BLOCK_K)):

        # Calculate the K indices for this iteration.

        k_offsets = k_tile * BLOCK_K + offsets_k


        # Construct pointers for A tile.
        # Expected pointer shape:
        # (BLOCK_M, BLOCK_K)

        a_ptrs = (
            A_ptr
            + offsets_m[:, None] * stride_am
            + k_offsets[None, :] * stride_ak
        )


        # Construct pointers for B tile.
        # Expected pointer shape:
        # (BLOCK_K, BLOCK_N)

        b_ptrs = (
            B_ptr
            + k_offsets[:, None] * stride_bk
            + offsets_n[None, :] * stride_bn
        )

        # Generate boundary masks.

        a_mask = (
            (offsets_m[:, None] < M)
            & (k_offsets[None, :] < K)
        )
        b_mask = (
            (k_offsets[:, None] < K)
            & (offsets_n[None, :] < N)
        )

        # Load A and B tiles with masked loads.

        a = tl.load(a_ptrs, mask=a_mask, other=0)
        b = tl.load(b_ptrs, mask=b_mask, other=0)


        # Perform matrix multiplication and accumulate.

        accumulator = tl.dot(a, b, accumulator)


    # --------------------------------------------------------
    # Step 5 — Store Output Tile
    # --------------------------------------------------------

    # Convert FP32 accumulator to FP16.

    c = accumulator.to(tl.float16)

    # Construct C pointers and boundary mask.

    c_ptrs = (
        C_ptr
        + offsets_m[:, None] * stride_cm
        + offsets_n[None, :] * stride_cn
    )

    c_mask = (
        (offsets_m[:, None] < M)
        & (offsets_n[None, :] < N)
    )


    # Store the output tile.
    tl.store(c_ptrs, c, mask=c_mask)

In [6]:
# ============================================================
# Launch Triton GEMM Kernel
# ============================================================

grid = (
    triton.cdiv(M, BLOCK_M),
    triton.cdiv(N, BLOCK_N),
)

compiled_kernel = gemm_kernel[grid](
    A, B, C,

    M, N, K,

    A.stride(0),
    A.stride(1),

    B.stride(0),
    B.stride(1),

    C.stride(0),
    C.stride(1),

    BLOCK_M=BLOCK_M,
    BLOCK_N=BLOCK_N,
    BLOCK_K=BLOCK_K,

    num_warps=NUM_WARPS,
)

torch.cuda.synchronize()

print("Kernel execution completed.")
print("Grid:", grid)

Kernel execution completed.
Grid: (8, 8)


In [7]:
# ============================================================
# GEMM Correctness Validation
# ============================================================

reference = torch.matmul(A, B)

torch.testing.assert_close(
    C,
    reference,
    rtol=1e-2,
    atol=1e-2,
)

max_error = torch.max(
    torch.abs(C.float() - reference.float())
)

print("Correctness: PASSED")
print("Maximum absolute error:", max_error.item())

Correctness: PASSED
Maximum absolute error: 0.03125


In [8]:
# ------------------------------------------------------------
# Boundary Test
# ------------------------------------------------------------

M2, N2, K2 = 250, 510, 253

A2 = torch.randn((M2, K2), device="cuda", dtype=torch.float16)
B2 = torch.randn((K2, N2), device="cuda", dtype=torch.float16)
C2 = torch.empty((M2, N2), device="cuda", dtype=torch.float16)

grid2 = (
    triton.cdiv(M2, BLOCK_M),
    triton.cdiv(N2, BLOCK_N),
)

gemm_kernel[grid2](
    A2, B2, C2,

    M2, N2, K2,

    *A2.stride(),
    *B2.stride(),
    *C2.stride(),

    BLOCK_M=BLOCK_M,
    BLOCK_N=BLOCK_N,
    BLOCK_K=BLOCK_K,

    num_warps=NUM_WARPS,
)

torch.cuda.synchronize()

reference2 = torch.matmul(A2, B2)

torch.testing.assert_close(
    C2,
    reference2,
    rtol=1e-2,
    atol=1e-2,
)

print("Boundary Masking: PASSED")

Boundary Masking: PASSED


## Experiment 1.3 — Compiler IR Extraction

### 1. Objective

The objective of this experiment is to extract and inspect the intermediate representations (IR) generated by the Triton compiler for the tiled GEMM kernel implemented in Experiment 1.2.

We will investigate the compilation pipeline from high-level Triton operations to NVIDIA GPU instructions, focusing on four compilation stages:

- Triton IR (TTIR)
- Triton GPU IR (TTGIR)
- LLVM IR
- NVIDIA PTX

This experiment establishes the foundation for understanding compiler transformations, GPU-specific lowering, and hardware-aware code generation.

### 2. Compilation Pipeline

The simplified Triton compilation pipeline is:

    Triton Python GEMM Kernel
                |
                v
         Triton IR (TTIR)
                |
                | High-Level Transformations
                v
       Triton GPU IR (TTGIR)
                |
                | GPU-Specific Lowering
                v
             LLVM IR
                |
                | NVIDIA Backend
                v
               PTX
                |
                | PTX Assembly
                v
         GPU Machine Code

Each stage represents the computation at a different level of abstraction.

### 3. Experimental Procedure

#### Step 1 — Inspect Compiled Kernel

Access the compiled Triton kernel object and identify the available compilation artifacts.

#### Step 2 — Extract Intermediate Representations

Extract and save the following files:

    artifacts/
    ├── gemm.ttir
    ├── gemm.ttgir
    ├── gemm.llir
    └── gemm.ptx

#### Step 3 — Identify Key Compiler Operations

Inspect each representation and identify important operations.

| IR Stage | Key Elements to Inspect |
|---|---|
| TTIR | SSA values, tensor operations, memory loads/stores, dot operations |
| TTGIR | GPU layouts, thread/warp mappings, layout conversions, dot operations |
| LLVM IR | Low-level instructions, control flow, NVVM intrinsics |
| PTX | Global memory instructions, predicates, matrix multiplication instructions |

The exact operation names and instruction sequences depend on the Triton version and target GPU architecture.

#### Step 4 — Document Initial Observations

Record the major differences between compilation stages, including changes in abstraction level, execution information, and hardware-specific operations.

### 4. Expected Outcomes

By completing this experiment, we should be able to:

1. Access compilation artifacts from a Triton JIT-compiled GEMM kernel.
2. Extract TTIR, TTGIR, LLVM IR, and PTX.
3. Identify important GEMM-related operations in each representation.
4. Understand the purpose of different IR levels.
5. Establish a collection of compiler artifacts for detailed analysis.

### 5. Success Criteria

- [ ] TTIR successfully extracted and saved.
- [ ] TTGIR successfully extracted and saved.
- [ ] LLVM IR successfully extracted and saved.
- [ ] PTX successfully extracted and saved.
- [ ] Key compiler operations identified.
- [ ] Initial observations documented.

### 6. Connection to Subsequent Experiments

The extracted IR files will be used in the following experiments:

**Experiment 1.4 — TTIR Analysis**

Analyze high-level tensor operations, SSA representation, and control flow.

**Experiment 1.5 — TTGIR Analysis**

Investigate GPU layouts, thread/warp mapping, and GPU-specific compiler transformations.

**Experiment 1.6 — LLVM IR & PTX Analysis**

Examine low-level code generation and trace GEMM operations to NVIDIA GPU instructions.

The ultimate goal is to understand how a high-level `tl.dot` operation is progressively lowered into hardware-specific matrix computation instructions.

**Next: Extract and inspect the compiled GEMM kernel artifacts.**

### Inspect Compiled Kernel

In [9]:
# ============================================================
# Experiment 1.3 — Compiler IR Extraction
# Step 1: Inspect Compiled Kernel
# ============================================================

# 1. Inspect the compiled kernel object

print("Compiled Kernel:", compiled_kernel)
print("Kernel Type:", type(compiled_kernel))


# 2. Retrieve available compiler artifacts

asm_dict = compiled_kernel.asm

print("\nAvailable Compiler Artifacts:")

for key in asm_dict.keys():
    print(" -", key)

Compiled Kernel: <triton.compiler.compiler.CompiledKernel object at 0x7a1cdc1dda90>
Kernel Type: <class 'triton.compiler.compiler.CompiledKernel'>

Available Compiler Artifacts:
 - source
 - ttir
 - ttgir
 - llir
 - ptx
 - cubin


### Extract and Save IR Files

In [10]:
# ============================================================
# Step 2: Extract Intermediate Representations
# ============================================================

from pathlib import Path

# Create output directory
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

# IR stages to extract
ir_stages = {
    "ttir": "gemm.ttir",
    "ttgir": "gemm.ttgir",
    "llir": "gemm.llir",
    "ptx": "gemm.ptx",
}

for stage, filename in ir_stages.items():
    # Check whether the IR stage is available.
    if stage not in asm_dict:
        print(f"Skipping {stage}: Not available")
        continue

    # Retrieve the IR content.

    ir_content = asm_dict[stage]

    # Construct the output path.

    output_path =  artifact_dir / filename

    # Save IR content to a text file.
    output_path.write_text(str(ir_content), encoding="utf-8")

    print(f"Saved: {output_path}")


print("\nIR extraction completed.")

Saved: artifacts/gemm.ttir
Saved: artifacts/gemm.ttgir
Saved: artifacts/gemm.llir
Saved: artifacts/gemm.ptx

IR extraction completed.


### Inspect IR Structure

In [11]:
# ============================================================
# Step 3: Inspect IR Structure
# ============================================================

for stage in ["ttir", "ttgir", "llir", "ptx"]:
    # Retrieve the IR content.
    ir_content = asm_dict[stage]

    # Split the content into lines.
    lines = str(ir_content).splitlines()

    print("\n" + "=" * 70)
    print(f"Compiler Stage: {stage.upper()}")
    print("=" * 70)

    # Print the first 40 lines.

    for line in lines[:40]:
        print(line)

    # Print total line count.
    print("\nTotal Lines:", len(lines))


Compiler Stage: TTIR
#loc = loc("/tmp/ipykernel_5713/411635098.py":2:0)
#loc45 = loc("A_ptr"(#loc))
#loc46 = loc("B_ptr"(#loc))
#loc47 = loc("C_ptr"(#loc))
module {
  tt.func public @gemm_kernel(%A_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("A_ptr"(#loc)), %B_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("B_ptr"(#loc)), %C_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("C_ptr"(#loc))) attributes {noinline = false} {
    %c8_i32 = arith.constant 8 : i32 loc(#loc1)
    %cst = arith.constant dense<0.000000e+00> : tensor<32x64xf16> loc(#loc1)
    %cst_0 = arith.constant dense<0.000000e+00> : tensor<32x32xf16> loc(#loc1)
    %c1_i32 = arith.constant 1 : i32 loc(#loc2)
    %c0_i32 = arith.constant 0 : i32 loc(#loc2)
    %cst_1 = arith.constant dense<512> : tensor<1x64xi32> loc(#loc1)
    %cst_2 = arith.constant dense<256> : tensor<1x32xi32> loc(#loc1)
    %cst_3 = arith.constant dense<512> : tensor<32x1xi32> loc(#loc1)
    %cst_4 = arith.constant dense<256> : tensor<32x1xi3

### Identify Key Compiler Operations

In [12]:
# ============================================================
# Step 4: Identify Key Compiler Operations
# ============================================================

keywords = {
    "ttir": [
        "tt.load",
        "tt.store",
        "dot",
        "scf.for",
    ],

    "ttgir": [
        "blocked",
        "dot",
        "convert_layout",
        "warpsPerCTA",
    ],

    "llir": [
        "getelementptr",
        "phi",
        "nvvm",
        "asm",
    ],

    "ptx": [
        "ld.global",
        "st.global",
        "mma.sync",
    ],
}


for stage, search_terms in keywords.items():

    print("\n" + "=" * 70)
    print(f"Analyzing: {stage.upper()}")
    print("=" * 70)

    # Retrieve the IR text.
    ir_content = asm_dict[stage]

    # Split into lines.

    lines = str(ir_content).splitlines()

    for keyword in search_terms:
        # Search for lines containing the keyword.
        matches = [
            line for line in lines
            if keyword in line
        ]

        print(f"\nKeyword: {keyword}")
        print(f"Matches: {len(matches)}")

        # TODO 4:
        # Print at most the first 5 matches.

        for line in matches[:5]:
            print("  ", line.strip())


Analyzing: TTIR

Keyword: tt.load
Matches: 2
   %a = tt.load %a_ptrs_32, %a_mask_43, %cst_0 : tensor<32x32x!tt.ptr<f16>> loc(#loc76)
   %b = tt.load %b_ptrs_39, %b_mask_47, %cst : tensor<32x64x!tt.ptr<f16>> loc(#loc77)

Keyword: tt.store
Matches: 1
   tt.store %c_ptrs_18, %c, %c_mask_22 : tensor<32x64x!tt.ptr<f16>> loc(#loc43)

Keyword: dot
Matches: 1
   %accumulator_48 = tt.dot %a, %b, %accumulator_23, inputPrecision = tf32 : tensor<32x32xf16> * tensor<32x64xf16> -> tensor<32x64xf32> loc(#loc78)

Keyword: scf.for
Matches: 1
   %accumulator_11 = scf.for %k_tile = %c0_i32 to %c8_i32 step %c1_i32 iter_args(%accumulator_23 = %accumulator) -> (tensor<32x64xf32>)  : i32 {

Analyzing: TTGIR

Keyword: blocked
Matches: 74
   #blocked = #ttg.blocked<{sizePerThread = [1, 8], threadsPerWarp = [4, 8], warpsPerCTA = [4, 1], order = [1, 0]}>
   #blocked1 = #ttg.blocked<{sizePerThread = [1, 8], threadsPerWarp = [8, 4], warpsPerCTA = [4, 1], order = [1, 0]}>
   #blocked2 = #ttg.blocked<{sizePerThread

## Experiment 1.4 — Triton GPU IR (TTGIR) Analysis

### 1. Objective

The objective of this experiment is to investigate how the Triton compiler transforms high-level tensor computations into GPU-specific intermediate representations.

Building on the TTIR analysis, we will examine the generated Triton GPU IR (TTGIR), focusing on tensor layouts, thread and warp mapping, memory organization, and matrix multiplication lowering.

The primary goal is to understand how logical tensor operations are distributed across GPU execution resources.

### 2. Background

TTIR describes the logical computation of a GPU kernel, including tensor operations, memory accesses, and structured control flow.

TTGIR introduces hardware-aware information that specifies how tensor elements are distributed across GPU threads and warps.

The transition can be summarized as:

    TTIR
     |
     | Tensor-level operations
     | SSA and structured control flow
     v
    TTGIR
     |
     | Tensor layout assignment
     | Thread and warp mapping
     | GPU-specific memory operations
     v
    Lower-Level IR

Both TTIR and TTGIR use MLIR-based representations, but TTGIR provides more explicit GPU execution information.

### 3. Experimental Configuration

The experiment uses the GEMM kernel compiled in Experiment 1.2.

| Parameter | Value |
|---|---|
| GPU | NVIDIA Tesla T4 |
| Compute Capability | 7.5 |
| Triton Version | 3.6.0 |
| Input Data Type | FP16 |
| Accumulator Type | FP32 |
| BLOCK_M | 32 |
| BLOCK_N | 64 |
| BLOCK_K | 32 |
| num_warps | 4 |

The previously extracted `gemm.ttir` and `gemm.ttgir` files will be used for analysis.

### 4. Experimental Procedure

#### Step 1 — Inspect TTGIR Module Attributes

Examine the target GPU architecture and execution configuration.

Focus on:

- `ttg.target`
- `ttg.num-warps`
- `ttg.threads-per-warp`
- `ttg.num-ctas`

Verify how the compiler represents the target architecture and the number of warps assigned to each CTA.

#### Step 2 — Analyze Blocked Tensor Layouts

Identify and analyze the blocked layouts generated by the compiler.

For example:

    #ttg.blocked<{
        sizePerThread = [1, 8],
        threadsPerWarp = [4, 8],
        warpsPerCTA = [4, 1],
        order = [1, 0]
    }>

Investigate the meaning of:

- `sizePerThread`
- `threadsPerWarp`
- `warpsPerCTA`
- `order`

Determine how these parameters describe the distribution of tensor elements across threads and warps.

#### Step 3 — Compare TTIR and TTGIR Tensor Representations

Compare corresponding tensor operations across TTIR and TTGIR.

Focus on:

- Tensor shapes and element types
- GPU layout attributes
- Memory load/store operations
- SSA values and their dependencies
- Additional GPU-specific operations

Identify what information is introduced during GPU layout assignment and lowering.

#### Step 4 — Analyze Dot Operand Layouts

Investigate how the GEMM dot-product operands are represented in TTGIR.

Focus on:

- `tt.dot`
- `#ttg.dot_op`
- Dot operand tensor shapes
- Operand data types
- Accumulator data types
- Result layouts

Compare the dot-product representation in TTIR and TTGIR.

Pay particular attention to the observed transformation from FP16 operands in TTIR to FP32 operands in the extracted TTGIR.

The reason for this transformation will be investigated rather than assumed.

#### Step 5 — Investigate Shared Memory Operations

Identify shared-memory-related operations and layouts.

Focus on:

- `ttg.local_alloc`
- `ttg.local_load`
- Shared memory descriptors
- Shared memory layout attributes
- Synchronization requirements

Investigate why shared memory is introduced and how it may support tensor data redistribution or GPU computation.

#### Step 6 — Analyze Layout Conversion

Identify `ttg.convert_layout` operations.

Investigate:

- Source tensor layout
- Destination tensor layout
- Whether tensor shapes and data types change
- Why layout conversion may be necessary
- Potential performance costs

Determine whether a layout conversion changes the distribution of data across GPU threads while preserving the logical tensor values.

### 5. Key Technical Questions

By completing this experiment, we aim to answer:

1. What is the difference between TTIR and TTGIR?
2. Why does Triton introduce GPU-specific tensor layouts?
3. What do `sizePerThread`, `threadsPerWarp`, and `warpsPerCTA` represent?
4. How are logical tensor elements distributed across GPU threads?
5. Why can a GEMM kernel require multiple tensor layouts?
6. What is a Dot Operand Layout?
7. Why does the compiler introduce shared memory operations?
8. What happens during a layout conversion?
9. How does TTGIR prepare tensor operations for lower-level GPU code generation?
10. Why do the dot-product operand types differ between the extracted TTIR and TTGIR?

### 6. Expected Outcomes

Upon completing this experiment, we should be able to:

- Explain the role of TTGIR in the Triton compilation pipeline.
- Interpret blocked tensor layout attributes.
- Understand thread and warp distribution for selected tensors.
- Identify dot operand layouts and related memory operations.
- Trace important GEMM operations from TTIR to TTGIR.
- Explain the purpose and potential cost of layout conversions.
- Identify hardware-specific lowering decisions that require further investigation.

### 7. Success Criteria

- [ ] TTGIR module attributes identified.
- [ ] Blocked tensor layouts analyzed.
- [ ] Thread and warp mappings explained.
- [ ] TTIR and TTGIR operations compared.
- [ ] Dot operand layouts identified.
- [ ] Shared memory operations analyzed.
- [ ] Layout conversion operations investigated.
- [ ] FP16-to-FP32 dot operand transformation documented.
- [ ] Key findings summarized.

### 8. Connection to the Next Experiment

The TTGIR analysis provides the foundation for investigating lower-level GPU code generation.

In the next experiment, we will examine LLVM IR and PTX to understand how GPU-specific operations are lowered into executable instructions.

Special attention will be given to:

- NVVM intrinsics
- Global and shared memory instructions
- Thread and block indexing
- Synchronization instructions
- FP32 FMA and Tensor Core matrix instructions

**Next: Experiment 1.5 — LLVM IR and PTX Analysis**

In [13]:
# ============================================================
# Experiment 1.4 — TTGIR Analysis
# Step 1: Inspect TTGIR Module Attributes
# ============================================================

# Retrieve TTGIR
ttgir = asm_dict["ttgir"]

# Split into lines
lines = str(ttgir).splitlines()

# Module attributes to inspect
attributes = [
    "ttg.target",
    "ttg.num-warps",
    "ttg.threads-per-warp",
    "ttg.num-ctas",
]

for attr in attributes:

    # Find lines containing the attribute
    matches = [
        line for line in lines
        if attr in line
    ]

    print(f"\nAttribute: {attr}")

    # Print matching lines
    for line in matches:
        print(" ", line.strip())


Attribute: ttg.target
  module attributes {"ttg.num-ctas" = 1 : i32, "ttg.num-warps" = 4 : i32, ttg.target = "cuda:75", "ttg.threads-per-warp" = 32 : i32} {

Attribute: ttg.num-warps
  module attributes {"ttg.num-ctas" = 1 : i32, "ttg.num-warps" = 4 : i32, ttg.target = "cuda:75", "ttg.threads-per-warp" = 32 : i32} {

Attribute: ttg.threads-per-warp
  module attributes {"ttg.num-ctas" = 1 : i32, "ttg.num-warps" = 4 : i32, ttg.target = "cuda:75", "ttg.threads-per-warp" = 32 : i32} {

Attribute: ttg.num-ctas
  module attributes {"ttg.num-ctas" = 1 : i32, "ttg.num-warps" = 4 : i32, ttg.target = "cuda:75", "ttg.threads-per-warp" = 32 : i32} {


In [14]:
# ============================================================
# Step 2: Analyze Blocked Tensor Layouts
# ============================================================
import re
# 1. Find blocked layout definitions
blocked_layouts = [
    line for line in lines
    if "#ttg.blocked<" in line
    and line.strip().startswith("#")
]

print("Number of Blocked Layouts:", len(blocked_layouts))

# 2. Print each blocked layout
for layout in blocked_layouts:
    print(layout.strip())

# Identify the following layout attributes:
#
# sizePerThread
# threadsPerWarp
# warpsPerCTA
# order
attributes = [
    "sizePerThread",
    "threadsPerWarp",
    "warpsPerCTA",
    "order",
]

print("\n" + "=" * 70)
print("Blocked Layout Attributes")
print("=" * 70)

for layout in blocked_layouts:

    layout_name = layout.split("=")[0].strip()

    print(f"\nLayout: {layout_name}")

    for attr in attributes:

        pattern = rf"\b{attr}\s*=\s*(\[[^\]]+\])"

        match = re.search(pattern, layout)

        if match:
            print(f"  {attr:16s}: {match.group(1)}")

Number of Blocked Layouts: 3
#blocked = #ttg.blocked<{sizePerThread = [1, 8], threadsPerWarp = [4, 8], warpsPerCTA = [4, 1], order = [1, 0]}>
#blocked1 = #ttg.blocked<{sizePerThread = [1, 8], threadsPerWarp = [8, 4], warpsPerCTA = [4, 1], order = [1, 0]}>
#blocked2 = #ttg.blocked<{sizePerThread = [4, 4], threadsPerWarp = [2, 16], warpsPerCTA = [4, 1], order = [1, 0]}>

Blocked Layout Attributes

Layout: #blocked
  sizePerThread   : [1, 8]
  threadsPerWarp  : [4, 8]
  warpsPerCTA     : [4, 1]
  order           : [1, 0]

Layout: #blocked1
  sizePerThread   : [1, 8]
  threadsPerWarp  : [8, 4]
  warpsPerCTA     : [4, 1]
  order           : [1, 0]

Layout: #blocked2
  sizePerThread   : [4, 4]
  threadsPerWarp  : [2, 16]
  warpsPerCTA     : [4, 1]
  order           : [1, 0]


In [15]:
# ============================================================
# Step 3: Compare TTIR and TTGIR Operations
# ============================================================

# 1. Retrieve TTIR
ttir = asm_dict["ttir"]
ttgir = asm_dict["ttgir"]

# 2. Split TTIR and TTGIR into lines
ttir_lines = str(ttir).splitlines()
ttgir_lines = str(ttgir).splitlines()

# 3. Operations to compare
operations = [
    "tt.load",
    "tt.store",
    "tt.dot",
    "tt.make_range",
]

for op in operations:

    # TODO 3: Find matching operations in TTIR
    ttir_matches = [
        line for line in ttir_lines
        if op in line
    ]

    # TODO 4: Find matching operations in TTGIR
    ttgir_matches = [
        line for line in ttgir_lines
        if op in line
    ]

    print("\n" + "=" * 70)
    print("Operation:", op)
    print("=" * 70)

    print("\nTTIR:")
    for line in ttir_matches[:3]:
        print(" ", line.strip())

    print("\nTTGIR:")
    for line in ttgir_matches[:3]:
        print(" ", line.strip())


Operation: tt.load

TTIR:
  %a = tt.load %a_ptrs_32, %a_mask_43, %cst_0 : tensor<32x32x!tt.ptr<f16>> loc(#loc76)
  %b = tt.load %b_ptrs_39, %b_mask_47, %cst : tensor<32x64x!tt.ptr<f16>> loc(#loc77)

TTGIR:
  %a = tt.load %a_ptrs_40, %a_mask_48, %cst_4 : tensor<32x32x!tt.ptr<f16>, #blocked1> loc(#loc72)
  %b = tt.load %b_ptrs_45, %b_mask_51, %cst_5 : tensor<32x64x!tt.ptr<f16>, #blocked> loc(#loc73)

Operation: tt.store

TTIR:
  tt.store %c_ptrs_18, %c, %c_mask_22 : tensor<32x64x!tt.ptr<f16>> loc(#loc43)

TTGIR:
  tt.store %c_ptrs_29, %0, %c_mask_31 : tensor<32x64x!tt.ptr<f16>, #blocked> loc(#loc40)

Operation: tt.dot

TTIR:
  %accumulator_48 = tt.dot %a, %b, %accumulator_23, inputPrecision = tf32 : tensor<32x32xf16> * tensor<32x64xf16> -> tensor<32x64xf32> loc(#loc78)

TTGIR:
  %accumulator_58 = tt.dot %accumulator_54, %accumulator_57, %accumulator_33, inputPrecision = tf32 : tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> * tensor<32x64xf32, #ttg.dot_op<{opIdx = 1, par

In [16]:
# ============================================================
# Optional — Inspect Operation Context
# ============================================================

def show_operation_context(ir_lines, keyword, context=5):

    for i, line in enumerate(ir_lines):

        if keyword in line:

            start = max(0, i - context)
            end = min(len(ir_lines), i + context + 1)

            print("\n" + "=" * 70)
            print(f"Operation: {keyword} | Line: {i + 1}")
            print("=" * 70)

            for j in range(start, end):
                print(f"{j + 1:4d}: {ir_lines[j]}")


# Example: Examine the dot operation in TTGIR
show_operation_context(
    ttgir_lines,
    keyword="tt.dot",
    context=8,
)


Operation: tt.dot | Line: 79
  71:       %a = tt.load %a_ptrs_40, %a_mask_48, %cst_4 : tensor<32x32x!tt.ptr<f16>, #blocked1> loc(#loc72)
  72:       %b = tt.load %b_ptrs_45, %b_mask_51, %cst_5 : tensor<32x64x!tt.ptr<f16>, #blocked> loc(#loc73)
  73:       %accumulator_52 = arith.extf %a : tensor<32x32xf16, #blocked1> to tensor<32x32xf32, #blocked1> loc(#loc74)
  74:       %accumulator_53 = ttg.local_alloc %accumulator_52 : (tensor<32x32xf32, #blocked1>) -> !ttg.memdesc<32x32xf32, #shared, #smem> loc(#loc74)
  75:       %accumulator_54 = ttg.local_load %accumulator_53 : !ttg.memdesc<32x32xf32, #shared, #smem> -> tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> loc(#loc74)
  76:       %accumulator_55 = arith.extf %b : tensor<32x64xf16, #blocked> to tensor<32x64xf32, #blocked> loc(#loc74)
  77:       %accumulator_56 = ttg.local_alloc %accumulator_55 : (tensor<32x64xf32, #blocked>) -> !ttg.memdesc<32x64xf32, #shared, #smem> loc(#loc74)
  78:       %accumulator_57 = ttg.loca

### Key Findings — GEMM Dot Operand Lowering

The TTGIR analysis reveals a clear transformation of the GEMM dot operands.

1. The compiler initially loads FP16 matrix tiles using `tt.load`.
2. Both input tiles are explicitly converted from FP16 to FP32 using `arith.extf`.
3. The FP32 tensors are passed through `ttg.local_alloc` and `ttg.local_load`, introducing shared-memory-related data redistribution.
4. The resulting tensors use `#ttg.dot_op` layouts associated with the GEMM computation.
5. The `tt.dot` operation consumes FP32 operands and produces an FP32 accumulator.
6. The accumulator is carried across loop iterations using `scf.yield`.
7. After the GEMM reduction, the final accumulator is converted to FP16 using `arith.truncf`.

These observations show how TTGIR introduces explicit data type conversions, GPU-specific tensor layouts, and shared memory operations.

An important remaining question is whether the generated GEMM uses Tensor Core instructions or a SIMT-based FP32 arithmetic path. This will be investigated through LLVM IR and PTX inspection.

In [17]:
# ============================================================
# Step 4: Analyze Dot Operand Layouts
# ============================================================

# ------------------------------------------------------------
# 1. Identify Dot-related Operations
# ------------------------------------------------------------

dot_keywords = [
    "tt.dot",
    "ttg.dot_op",
    "ttg.local_load",
]

for keyword in dot_keywords:

    # 1. Find matching lines
    matches = [
        line for line in ttgir_lines
        if keyword in line
    ]

    print("\n" + "=" * 70)
    print("Keyword:", keyword)
    print("=" * 70)

    print(f"Number of matches: {len(matches)}")
    # 2. Print matching lines
    for line in matches:
        print(line.strip())

# ------------------------------------------------------------
# 2. Compare Dot Operations in TTIR and TTGIR
# TTIR:  FP16 x FP16 -> FP32
# TTGIR: FP32 x FP32 -> FP32
#
# Investigate why the types differ.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Dot Operation Comparison: TTIR vs TTGIR")
print("=" * 70)

ttir_dot = [
    line.strip()
    for line in ttir_lines
    if "tt.dot " in line
]

ttgir_dot = [
    line.strip()
    for line in ttgir_lines
    if "tt.dot " in line
]

print("\nTTIR Dot Operations:")
for line in ttir_dot:
    print(line)

print("\nTTGIR Dot Operations:")
for line in ttgir_dot:
    print(line)


# ------------------------------------------------------------
# 3. Inspect Dot Operation Context
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TTGIR Dot Operation Context")
print("=" * 70)

for i, line in enumerate(ttgir_lines):

    if "tt.dot " in line:

        start = max(0, i - 10)
        end = min(len(ttgir_lines), i + 6)

        for j in range(start, end):
            print(f"{j + 1:4d}: {ttgir_lines[j]}")

# ------------------------------------------------------------
# 4. Inspect Operand Data Types and Layouts
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Dot Operand Types and Layouts")
print("=" * 70)

import re

for stage, dot_lines in [
    ("TTIR", ttir_dot),
    ("TTGIR", ttgir_dot),
]:

    print(f"\nStage: {stage}")

    for line in dot_lines:

        # Extract tensor type descriptions
        tensor_types = re.findall(
            r"tensor<[^>]+>",
            line
        )

        for index, tensor_type in enumerate(tensor_types):
            print(f"  Tensor {index + 1}: {tensor_type}")


Keyword: tt.dot
Number of matches: 1
%accumulator_58 = tt.dot %accumulator_54, %accumulator_57, %accumulator_33, inputPrecision = tf32 : tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> * tensor<32x64xf32, #ttg.dot_op<{opIdx = 1, parent = #blocked2}>> -> tensor<32x64xf32, #blocked2> loc(#loc74)

Keyword: ttg.dot_op
Number of matches: 3
%accumulator_54 = ttg.local_load %accumulator_53 : !ttg.memdesc<32x32xf32, #shared, #smem> -> tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> loc(#loc74)
%accumulator_57 = ttg.local_load %accumulator_56 : !ttg.memdesc<32x64xf32, #shared, #smem> -> tensor<32x64xf32, #ttg.dot_op<{opIdx = 1, parent = #blocked2}>> loc(#loc74)
%accumulator_58 = tt.dot %accumulator_54, %accumulator_57, %accumulator_33, inputPrecision = tf32 : tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> * tensor<32x64xf32, #ttg.dot_op<{opIdx = 1, parent = #blocked2}>> -> tensor<32x64xf32, #blocked2> loc(#loc74)

Keyword: ttg.local_load
Number 

### Key Findings — TTGIR Dot Operand Analysis

The TTGIR analysis reveals several important transformations in the GEMM compilation pipeline.

1. The input matrix tiles are loaded as FP16 tensors with GPU-specific blocked layouts.
2. Both input tiles are explicitly converted from FP16 to FP32 using `arith.extf`.
3. Shared memory operations are introduced through `ttg.local_alloc` and `ttg.local_load`.
4. The compiler assigns dedicated dot operand layouts using `#ttg.dot_op`, with `opIdx = 0` for A and `opIdx = 1` for B.
5. The GEMM dot operation consumes FP32 operands and produces an FP32 accumulator.
6. The accumulator is carried across iterations using `scf.yield`, and the final result is converted to FP16 using `arith.truncf`.

These observations demonstrate how TTGIR introduces explicit data type transformations, tensor layouts, and memory organization for GPU execution.

The next step is to inspect LLVM IR and PTX to determine how the FP32 dot operation is implemented on the NVIDIA Tesla T4.

In [18]:
# ============================================================
# Investigate GEMM Arithmetic Instructions in PTX
# ============================================================

ptx_lines = str(asm_dict["ptx"]).splitlines()

keywords = [
    "mma.sync",
    "wmma",
    "fma.rn.f32",
    "fma.f32",
    "mul.f32",
    "add.f32",
    "ld.shared",
    "st.shared",
    "bar.sync",
]

for keyword in keywords:
    matches = [
        line.strip()
        for line in ptx_lines
        if keyword in line
        and not line.lstrip().startswith("//")
    ]

    print(f"\n{keyword}: {len(matches)} matches")

    for line in matches[:3]:
        print(" ", line)


mma.sync: 0 matches

wmma: 0 matches

fma.rn.f32: 512 matches
  fma.rn.f32 	%r320, %r182, %r195, %r867;
  fma.rn.f32 	%r321, %r172, %r195, %r866;
  fma.rn.f32 	%r322, %r162, %r195, %r865;

fma.f32: 0 matches

mul.f32: 0 matches

add.f32: 0 matches

ld.shared: 116 matches
  ld.shared.b32 	%r48, [%r10+80];
  ld.shared.b32 	%r49, [%r10+124];
  ld.shared.b32 	%r50, [%r10+208];

st.shared: 10 matches
  st.shared.v4.b32 	[%r9], {%r43, %r42, %r41, %r40};
  st.shared.v4.b32 	[%r9+16], {%r47, %r46, %r45, %r44};
  st.shared.v4.b32 	[%r9], {%r59, %r58, %r57, %r56};

bar.sync: 6 matches
  bar.sync 	0;
  bar.sync 	0;
  bar.sync 	0;


### Key Findings — PTX Code Generation

The PTX analysis reveals that the compiled GEMM uses a SIMT-based FP32 FMA implementation rather than Tensor Core MMA instructions.

**Observations:**

- 512 static `fma.rn.f32` instructions were identified.
- No `mma.sync` or WMMA instructions were found.
- The generated PTX contains 116 shared-memory load instructions and 10 shared-memory store instructions.
- Six CTA synchronization instructions (`bar.sync`) were identified.

**Compiler Pipeline Analysis:**

1. TTIR represents GEMM using FP16 operands and FP32 accumulation.
2. TTGIR explicitly converts both operands from FP16 to FP32.
3. Shared-memory operations and GPU-specific dot operand layouts are introduced.
4. The final PTX implements the dot computation using FP32 FMA instructions.

**Conclusion:**

Although NVIDIA Tesla T4 supports FP16 Tensor Cores, the current Triton GEMM compilation follows a SIMT FP32 arithmetic path.

The exact compiler decision responsible for selecting this implementation remains to be investigated.

This experiment demonstrates the importance of inspecting generated compiler IR and GPU instructions rather than assuming that high-level matrix operations automatically use specialized hardware.

In [19]:
# ============================================================
# Step 5: Inspect Shared Memory Operations
# ============================================================

shared_keywords = [
    "ttg.local_alloc",
    "ttg.local_load",
    "ttg.local_store",
    "ttg.swizzled_shared",
    "ttg.memdesc",
]

for keyword in shared_keywords:

    # Search matching lines
    matches = [
        line for line in ttgir_lines
        if keyword in line
    ]

    print(f"\n{keyword}: {len(matches)} matches")

    # Print at most the first 5 matching lines
    for line in matches[:5]:
        print(" ", line.strip())


ttg.local_alloc: 2 matches
  %accumulator_53 = ttg.local_alloc %accumulator_52 : (tensor<32x32xf32, #blocked1>) -> !ttg.memdesc<32x32xf32, #shared, #smem> loc(#loc74)
  %accumulator_56 = ttg.local_alloc %accumulator_55 : (tensor<32x64xf32, #blocked>) -> !ttg.memdesc<32x64xf32, #shared, #smem> loc(#loc74)

ttg.local_load: 2 matches
  %accumulator_54 = ttg.local_load %accumulator_53 : !ttg.memdesc<32x32xf32, #shared, #smem> -> tensor<32x32xf32, #ttg.dot_op<{opIdx = 0, parent = #blocked2}>> loc(#loc74)
  %accumulator_57 = ttg.local_load %accumulator_56 : !ttg.memdesc<32x64xf32, #shared, #smem> -> tensor<32x64xf32, #ttg.dot_op<{opIdx = 1, parent = #blocked2}>> loc(#loc74)

ttg.local_store: 0 matches

ttg.swizzled_shared: 1 matches
  #shared = #ttg.swizzled_shared<{vec = 1, perPhase = 1, maxPhase = 1, order = [1, 0]}>

ttg.memdesc: 4 matches
  %accumulator_53 = ttg.local_alloc %accumulator_52 : (tensor<32x32xf32, #blocked1>) -> !ttg.memdesc<32x32xf32, #shared, #smem> loc(#loc74)
  %accumul

In [20]:
# ============================================================
# Step 6: Analyze Layout Conversion
# ============================================================
import re
# ------------------------------------------------------------
# 1. Find Layout Conversion Operations
# ------------------------------------------------------------
conversions = [
    line for line in ttgir_lines
    if "ttg.convert_layout" in line
]

print("Total Layout Conversions:", len(conversions))

# ------------------------------------------------------------
# 2. Print Layout Conversion Operations
# ------------------------------------------------------------
for conversion in conversions:
    print(conversion.strip())


# ------------------------------------------------------------
# 3. Analyze Layout Conversion Attributes
# For each conversion, identify:
#
# 1. Source layout
# 2. Destination layout
# 3. Tensor shape
# 4. Element data type
#
# Does the shape change?
# Does the element type change?
# What changes during conversion?
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Layout Conversion Analysis")
print("=" * 70)

for i, conversion in enumerate(conversions):

    print(f"\nConversion {i + 1}")

    # Extract source and destination tensor types
    tensor_types = re.findall(
        r"tensor<[^>]+>",
        conversion
    )

    if len(tensor_types) < 2:
        print("Unable to extract source/destination types.")
        continue

    source_type = tensor_types[0]
    destination_type = tensor_types[1]

    # Extract layouts
    source_layout = re.search(
        r"#blocked\d*",
        source_type
    )

    destination_layout = re.search(
        r"#blocked\d*",
        destination_type
    )

    # Extract tensor shape and element type
    source_info = re.search(
        r"tensor<([\dx]+)x(f\d+|i\d+)",
        source_type
    )

    destination_info = re.search(
        r"tensor<([\dx]+)x(f\d+|i\d+)",
        destination_type
    )

    print("Source Type:", source_type)
    print("Destination Type:", destination_type)

    if source_layout and destination_layout:
        print("Source Layout:", source_layout.group())
        print("Destination Layout:", destination_layout.group())

        print(
            "Layout Changed:",
            source_layout.group() != destination_layout.group()
        )

    if source_info and destination_info:

        source_shape, source_dtype = source_info.groups()
        dest_shape, dest_dtype = destination_info.groups()

        print("Source Shape:", source_shape)
        print("Destination Shape:", dest_shape)

        print("Source Data Type:", source_dtype)
        print("Destination Data Type:", dest_dtype)

        print("Shape Changed:", source_shape != dest_shape)
        print("Data Type Changed:", source_dtype != dest_dtype)

Total Layout Conversions: 1
%0 = ttg.convert_layout %c : tensor<32x64xf16, #blocked2> -> tensor<32x64xf16, #blocked> loc(#loc40)

Layout Conversion Analysis

Conversion 1
Source Type: tensor<32x64xf16, #blocked2>
Destination Type: tensor<32x64xf16, #blocked>
Source Layout: #blocked2
Destination Layout: #blocked
Layout Changed: True
Source Shape: 32x64
Destination Shape: 32x64
Source Data Type: f16
Destination Data Type: f16
Shape Changed: False
Data Type Changed: False


### Key Findings — Layout Conversion

The TTGIR contains one explicit `ttg.convert_layout` operation.

The conversion transforms the output tensor from `#blocked2`
to `#blocked`, while preserving its 32x64 shape and FP16
element type.

The source layout uses:

- sizePerThread = [4, 4]
- threadsPerWarp = [2, 16]

The destination layout uses:

- sizePerThread = [1, 8]
- threadsPerWarp = [4, 8]

This demonstrates that GPU tensor layouts determine how
logical tensor elements are distributed across threads.

The compiler uses one layout for dot computation and
another for the subsequent output store.

Depending on the source and destination layouts, such a
conversion may require register rearrangement, warp-level
communication, or shared-memory exchange.

The exact lowering mechanism and performance overhead
should be verified by inspecting the generated PTX and
profiling the kernel.


# Experiment 1.5 — LLVM IR & PTX Analysis

## Objective

The objective of this experiment is to investigate how Triton lowers GPU-specific intermediate representations (TTGIR) into LLVM IR and NVIDIA PTX instructions.

In Experiment 1.4, we observed that the compiler:

- Converted FP16 GEMM operands into FP32 using `arith.extf`.
- Introduced GPU-specific blocked and dot operand layouts.
- Used shared memory operations to prepare dot operands.
- Converted the accumulator layout before storing the final output.

We also observed that the generated PTX contained FP32 FMA instructions rather than Tensor Core MMA instructions.

In this experiment, we will investigate how these compiler transformations are represented at lower levels.

## Compilation Pipeline

```text
Triton Python Kernel
        |
        v
       TTIR
  High-level tensor operations
        |
        v
       TTGIR
  GPU layouts and memory operations
        |
        v
      LLVM IR
  SSA, basic blocks, phi nodes,
  memory operations, NVVM intrinsics
        |
        v
       PTX
  GPU instructions, registers,
  memory accesses, synchronization
        |
        v
      CUBIN
  GPU machine code
```

## Experimental Environment

- GPU: NVIDIA Tesla T4 (SM75)
- Compiler: Triton 3.6.0
- Kernel: Tiled FP16 GEMM with FP32 accumulation
- Matrix dimensions: M=256, N=512, K=256
- Tile configuration: BLOCK_M=32, BLOCK_N=64, BLOCK_K=32
- Number of warps: 4

## Analysis Tasks

### Step 1 — Inspect LLVM IR Structure

Examine the generated LLVM IR and identify:

- Module and function definitions
- Basic blocks and control-flow instructions
- LLVM SSA values
- Branch and return instructions

### Step 2 — Analyze Phi Nodes and Loop Lowering

Investigate how structured loops in TTGIR are transformed into LLVM control flow.

Focus on:

- Loop induction variables
- Phi nodes
- Loop-carried accumulators
- Back edges and exit blocks

### Step 3 — Analyze LLVM Memory Address Spaces

Identify how LLVM IR represents GPU memory operations, particularly global memory and shared memory.

### Step 4 — Inspect NVVM Intrinsics

Locate GPU-specific intrinsics used for thread indexing, block indexing, and synchronization.

### Step 5 — Analyze PTX Instructions

Inspect the generated PTX to identify:

- FP32 FMA instructions
- Tensor Core MMA instructions, if present
- Global and shared memory accesses
- CTA synchronization instructions

### Step 6 — Cross-Stage Lowering Analysis

Connect the observations across TTGIR, LLVM IR, and PTX to understand the complete GEMM lowering process.

## Expected Learning Outcomes

By completing this experiment, we aim to:

1. Understand how structured GPU compiler IR is lowered into LLVM SSA and control flow.
2. Identify how GPU memory operations and synchronization appear in LLVM IR and PTX.
3. Explain how a high-level `tt.dot` operation becomes lower-level arithmetic instructions.
4. Distinguish logical tensor operations from GPU execution instructions.
5. Develop practical experience in tracing compiler-generated code across multiple IR levels.



## Step 1 — Inspect LLVM IR Structure

### Objective

The objective of this step is to understand the basic structure of the LLVM IR generated by the Triton compiler.

Unlike TTIR and TTGIR, which represent tensor operations and GPU-specific data layouts, LLVM IR uses a lower-level, SSA-based representation consisting of functions, basic blocks, instructions, and explicit control flow.

In this step, we will examine how our GEMM kernel is represented in LLVM IR before analyzing specific GPU instructions.

### Tasks

**1. Inspect the LLVM IR Module**

Retrieve the generated LLVM IR and examine its header to identify:

- Module-level information
- Target triple and data layout, if present
- Kernel function definitions
- Function attributes

**2. Identify Functions and Basic Blocks**

Locate LLVM function definitions and basic block labels.

A basic block is a sequence of instructions with a single entry point and a terminator that determines how control leaves the block.

**3. Analyze Control Flow Instructions**

Search for LLVM terminator instructions such as:

- `br` — Unconditional or conditional branch
- `ret` — Return from a function
- `switch` — Multi-way conditional branch
- `unreachable` — Indicates that control flow cannot legally reach this point

These instructions explicitly represent the control flow that was previously expressed using structured operations such as `scf.for` in TTIR and TTGIR.

### Key Concepts

- **LLVM Module:** A compilation unit containing functions, global values, and metadata.
- **LLVM Function:** A collection of basic blocks implementing a function or GPU kernel.
- **Basic Block:** A sequence of LLVM instructions ending with a terminator.
- **SSA (Static Single Assignment):** Each SSA value is defined once, making data dependencies explicit.
- **CFG (Control Flow Graph):** A graph in which basic blocks are nodes and control-flow transfers are edges.

### Questions to Investigate

1. How is the Triton GEMM kernel represented as an LLVM function?
2. How many basic blocks are present in the generated LLVM IR?
3. Which branch and return instructions appear?
4. Are high-level operations such as `tt.dot` and `ttg.local_load` still visible?
5. How does the LLVM IR representation differ from TTIR and TTGIR?

### Expected Outcome

By the end of this step, we should be able to identify the LLVM kernel function, recognize its basic blocks and control-flow instructions, and explain how LLVM IR provides a lower-level representation for subsequent GPU code generation.

This analysis will also prepare us for Step 2, where we will investigate how Triton's structured GEMM loop is lowered into LLVM basic blocks, branches, and phi nodes.


In [21]:
# ============================================================
# Experiment 1.5 — LLVM IR & PTX Analysis
# Step 1: Inspect LLVM IR Structure
# ============================================================

# Retrieve compiled LLVM IR and PTX
llir = str(asm_dict["llir"])
ptx = str(asm_dict["ptx"])

# Split into lines
llir_lines = llir.splitlines()
ptx_lines = ptx.splitlines()

print("LLVM IR Lines:", len(llir_lines))
print("PTX Lines:", len(ptx_lines))

# Show LLVM IR header
print("\n" + "=" * 70)
print("LLVM IR Header")
print("=" * 70)

for i, line in enumerate(llir_lines[:45], start=1):
    print(f"{i:4d}: {line}")

LLVM IR Lines: 610
PTX Lines: 954

LLVM IR Header
   1: ; ModuleID = 'LLVMDialectModule'
   2: source_filename = "LLVMDialectModule"
   3: target datalayout = "e-p3:32:32-p4:32:32-p5:32:32-p6:32:32-p7:32:32-i64:64-i128:128-i256:256-v16:16-v32:32-n16:32:64"
   4: 
   5: @global_smem = external local_unnamed_addr addrspace(3) global [0 x i8], align 16
   6: 
   7: ; Function Attrs: nounwind
   8: define ptx_kernel void @gemm_kernel(ptr addrspace(1) %0, ptr addrspace(1) %1, ptr addrspace(1) %2, ptr addrspace(1) readnone captures(none) %3, ptr addrspace(1) readnone captures(none) %4) local_unnamed_addr #0 !dbg !4 {
   9:   %6 = tail call i32 @llvm.nvvm.read.ptx.sreg.ctaid.x(), !dbg !7
  10:   %7 = tail call i32 @llvm.nvvm.read.ptx.sreg.ctaid.y(), !dbg !8
  11:   %8 = shl i32 %6, 5, !dbg !9
  12:   %9 = tail call i32 @llvm.nvvm.read.ptx.sreg.tid.x(), !dbg !10
  13:   %10 = lshr i32 %9, 2, !dbg !10
  14:   %11 = and i32 %10, 31, !dbg !10
  15:   %12 = lshr i32 %9, 3, !dbg !10
  16:   %13 = a

In [22]:
# ============================================================
# Step 1.2: LLVM Functions and Basic Blocks
# ============================================================

import re

# Find function definitions
functions = [
    line.strip()
    for line in llir_lines
    if re.match(r"^\s*define\b", line)
]

print("LLVM Function Definitions:", len(functions))

for line in functions:
    print(line)


# Find basic block labels
basic_blocks = [
    line.strip()
    for line in llir_lines
    if re.match(r"^\s*(?:[A-Za-z$._][\w$.-]*|\d+):(?:\s|$)", line)
]

print("\nBasic Block Labels:", len(basic_blocks))

for line in basic_blocks[:25]:
    print(line)

LLVM Function Definitions: 1
define ptx_kernel void @gemm_kernel(ptr addrspace(1) %0, ptr addrspace(1) %1, ptr addrspace(1) %2, ptr addrspace(1) readnone captures(none) %3, ptr addrspace(1) readnone captures(none) %4) local_unnamed_addr #0 !dbg !4 {

Basic Block Labels: 2
116:                                              ; preds = %5, %116
420:                                              ; preds = %116


In [23]:
# ============================================================
# Step 1.3: Inspect LLVM Control Flow
# ============================================================

terminators = [
    "br ",
    "ret ",
    "switch ",
    "unreachable",
]

for keyword in terminators:

    matches = [
        line.strip()
        for line in llir_lines
        if re.match(
            rf"^\s*{re.escape(keyword)}",
            line
        )
    ]

    print("\n" + "=" * 70)
    print(f"Terminator: {keyword}")
    print("=" * 70)

    print("Matches:", len(matches))

    for line in matches[:10]:
        print(line)


Terminator: br 
Matches: 2
br label %116, !dbg !19
br i1 %exitcond.not, label %420, label %116, !dbg !19

Terminator: ret 
Matches: 1
ret void, !dbg !34

Terminator: switch 
Matches: 0

Terminator: unreachable
Matches: 0



### Step 1 — Key Findings

The generated LLVM IR contains one GPU kernel function, `gemm_kernel`, represented using the NVIDIA `ptx_kernel` calling convention.

The function contains three basic blocks: an implicit entry block, a loop block, and an exit block.

The control-flow structure is:

Entry Block → Loop Block → Exit Block

The loop block also contains a self-loop back edge, representing repeated GEMM K-tile iterations.

The compiler lowers Triton program IDs and thread indexing into NVVM intrinsics, including:

- `llvm.nvvm.read.ptx.sreg.ctaid.x`
- `llvm.nvvm.read.ptx.sreg.ctaid.y`
- `llvm.nvvm.read.ptx.sreg.tid.x`

Global memory pointers use LLVM address space 1, while shared memory references use address space 3.

The LLVM IR also uses integer shifts, bitwise operations, and `getelementptr` instructions to calculate per-thread memory addresses.

These observations demonstrate how high-level Triton tensor indexing and structured control flow are lowered into GPU-specific LLVM instructions and explicit control-flow constructs.





## Step 2 — Analyze Phi Nodes & Loop Lowering

### Objective

The objective of this step is to investigate how Triton lowers structured loops from TTIR/TTGIR into LLVM IR using basic blocks, phi nodes, and branch instructions.

In the previous experiment, we observed that the GEMM reduction loop was represented using `scf.for`, `iter_args`, and `scf.yield`.

In LLVM IR, the same loop is represented using explicit control flow and SSA phi nodes.

We will trace the loop induction variable, loop-carried accumulator, back edge, and exit condition to understand this transformation.

### Background — GEMM Reduction Loop

Our Triton GEMM performs matrix multiplication by iterating over K-dimension tiles.

With the current configuration:

- K = 256
- BLOCK_K = 32
- Number of K-tile iterations = 8

The computation can be expressed as:

```text
accumulator = 0

for k_tile in range(8):
    accumulator += dot(A_tile[k_tile], B_tile[k_tile])
```

In TTIR/TTGIR, this loop is represented using structured control flow:

```mlir
%result = scf.for %k_tile = %c0 to %c8 step %c1
    iter_args(%acc = %initial_accumulator) {
    %next_acc = tt.dot %a, %b, %acc
    scf.yield %next_acc
}
```

This example is simplified to illustrate the loop-carried dependency.

### Task 1 — Identify LLVM Phi Nodes

Search the generated LLVM IR for `phi` instructions.

In our previous inspection, we identified two important phi nodes:

```llvm
%indvars.iv = phi i64
    [ 0, %5 ], [ %indvars.iv.next, %116 ]

%117 = phi <16 x float>
    [ zeroinitializer, %5 ], [ %419, %116 ]
```

Investigate the meaning of each incoming value and its predecessor basic block.

**Questions:**

1. Why does each phi node have two incoming values?
2. What happens when execution enters the loop for the first time?
3. What happens when execution returns through the loop back edge?
4. Why does the accumulator require a phi node?

### Task 2 — Trace the Loop Induction Variable

Locate the LLVM instructions defining:

- `%indvars.iv`
- `%indvars.iv.next`
- `%exitcond.not`

Determine how the compiler:

1. Initializes the loop index.
2. Updates the index after each iteration.
3. Evaluates the loop exit condition.

Verify that the generated loop performs the eight K-tile iterations required by the GEMM kernel.

### Task 3 — Analyze the Loop-Carried Accumulator

Inspect the phi node:

```llvm
%117 = phi <16 x float>
    [ zeroinitializer, %5 ], [ %419, %116 ]
```

Trace how the accumulator is updated inside the loop and how the updated value is passed into the next iteration.

Pay particular attention to the relationship between:

- The initial accumulator (`zeroinitializer`)
- The current accumulator (`%117`)
- The updated accumulator (`%419`)
- The loop back edge (`%116`)

The LLVM vector type `<16 x float>` represents 16 FP32 values. It is a lower-level representation of a portion of the logical GEMM accumulator.

### Task 4 — Reconstruct the LLVM Control Flow Graph

Based on Step 1, the generated LLVM function contains three basic blocks:

- Entry block: `%5`
- Loop block: `%116`
- Exit block: `%420`

The control flow can be represented as:

```text
       +----------------+
       | Entry Block %5 |
       +----------------+
                |
                v
       +----------------+
       | Loop Block     |<----+
       | %116           |     |
       |                |     |
       | Phi Nodes      |     |
       | GEMM Compute   |     |
       | Loop Update    |     |
       +----------------+     |
           |          |       |
        Exit       Continue --+
           |
           v
       +----------------+
       | Exit Block     |
       | %420           |
       +----------------+
                |
                v
           ret void
```

Inspect the following branch instruction:

```llvm
br i1 %exitcond.not, label %420, label %116
```

Identify the loop's exit edge and back edge.

### Task 5 — Compare TTGIR and LLVM IR

Compare the corresponding control-flow constructs:

| TTIR / TTGIR | LLVM IR |
|---|---|
| `scf.for` | Basic blocks and branches |
| Loop induction variable | Induction-variable phi node |
| `iter_args` | Loop-carried SSA values and phi nodes |
| `scf.yield` | Updated values on the loop back edge |
| Loop termination condition | Integer comparison and conditional branch |

Explain how structured control flow is transformed into explicit CFG-based control flow while preserving the original loop semantics.

### Questions to Investigate

1. What is the purpose of a phi node in SSA?
2. Why are phi nodes necessary for loop-carried variables?
3. How does the loop distinguish between its initial value and updated value?
4. How is `scf.yield` represented after lowering to LLVM IR?
5. How does the generated CFG represent the GEMM reduction loop?
6. Why can a structured loop be represented using only one LLVM loop block?

### Expected Outcome

By the end of this step, we should be able to:

1. Identify and explain the loop induction variable and accumulator phi nodes.
2. Trace SSA values across loop iterations.
3. Reconstruct the generated LLVM control-flow graph.
4. Explain how `scf.for`, `iter_args`, and `scf.yield` are lowered into LLVM IR.
5. Relate the LLVM loop structure to the original Triton GEMM computation.

This experiment provides practical experience with SSA, phi nodes, control-flow graphs, and loop lowering in a real GPU compiler.


### Task 1 — Identify LLVM Phi Nodes

In [24]:
# ============================================================
# Experiment 1.5 — LLVM IR & PTX Analysis
# Step 2.1: Identify LLVM Phi Nodes
# ============================================================

import re

#1. Find indices of lines containing a phi instruction
phi_indices = [
    i for i, line in enumerate(llir_lines)
    if re.search(r"\bphi\b", line)
    and not line.lstrip().startswith(";")
]

print("Number of Phi Nodes:", len(phi_indices))

#2. Print each phi node with surrounding context
for i in phi_indices:

    print("\n" + "=" * 70)
    print(f"Phi Node at Line {i + 1}")
    print("=" * 70)

    # Print 5 lines before and after
    start = max(0, i - 5)
    end = min(len(llir_lines), i + 6)

    for j in range(start, end):
        print(f"{j + 1:4d}: {llir_lines[j]}")

Number of Phi Nodes: 2

Phi Node at Line 124
 119:   %114 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 500
 120:   %115 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 504
 121:   br label %116, !dbg !19
 122: 
 123: 116:                                              ; preds = %5, %116
 124:   %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %116 ]
 125:   %117 = phi <16 x float> [ zeroinitializer, %5 ], [ %419, %116 ]
 126:   %118 = shl nuw nsw i64 %indvars.iv, 5, !dbg !20
 127:   %119 = or disjoint i64 %118, %85, !dbg !21
 128:   %120 = or disjoint i64 %118, %86, !dbg !21
 129:   %gep318 = getelementptr half, ptr addrspace(1) %invariant.gep317, i64 %118, !dbg !22

Phi Node at Line 125
 120:   %115 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 504
 121:   br label %116, !dbg !19
 122: 
 123: 116:                                              ; preds = %5, %116
 124:   %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %116 ]
 125:   %117 = phi


### Step 2.1 — Key Findings: LLVM Phi Nodes

The generated LLVM IR contains two phi nodes at the loop header.

**1. Loop Induction Variable**

```llvm
%indvars.iv = phi i64
    [ 0, %5 ], [ %indvars.iv.next, %116 ]
```

This phi node selects the initial loop index from the entry block or the updated index from the loop back edge.

**2. Loop-Carried Accumulator**

```llvm
%117 = phi <16 x float>
    [ zeroinitializer, %5 ], [ %419, %116 ]
```

This phi node selects the initial zero accumulator or the updated accumulator from the previous iteration.

The loop header has two predecessors: the entry block and the loop back edge.

These phi nodes preserve loop-carried dependencies while maintaining LLVM's static single assignment form.

The LLVM IR also uses shift instructions to compute K-tile offsets. For example, shifting the induction variable left by five bits corresponds to multiplying the tile index by `BLOCK_K = 32`.

This experiment demonstrates how Triton's structured GEMM loop is represented using LLVM SSA values and explicit control flow.


### Task 2 — Trace Loop Induction Variable

In [25]:
# ============================================================
# Step 2.2: Trace Loop Induction Variable
# ============================================================

loop_keywords = [
    "%indvars.iv =",
    "%indvars.iv.next =",
    "%exitcond.not =",
]

for keyword in loop_keywords:

    #1: Find matching lines
    matches = [
        (i, line)
        for i, line in enumerate(llir_lines)
        if keyword in line
    ]

    print("\n" + "=" * 70)
    print("Keyword:", keyword)
    print("=" * 70)

    #2: Print matching lines and their line numbers
    for i, line in matches:
        print(f"{i + 1:4d}: {line.strip()}")


Keyword: %indvars.iv =
 124: %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %116 ]

Keyword: %indvars.iv.next =
 443: %indvars.iv.next = add nuw nsw i64 %indvars.iv, 1, !dbg !19

Keyword: %exitcond.not =
 444: %exitcond.not = icmp eq i64 %indvars.iv.next, 8, !dbg !19


### Task 3 — Trace Loop-Carried Accumulator

In [26]:
# ============================================================
# Step 2.3: Trace Loop-Carried Accumulator
# ============================================================

accumulator_keywords = [
    "%117 =",
    "%419 =",
]

for keyword in accumulator_keywords:

    #1: Find SSA definitions
    matches = [
        i for i, line in enumerate(llir_lines)
        if keyword in line
    ]

    print("\n" + "=" * 70)
    print("SSA Value:", keyword)
    print("=" * 70)

    #2: Print 5 lines of context around each definition
    for i in matches:

        start = max(0, i - 5)
        end = min(len(llir_lines), i + 6)

        for j in range(start, end):
            print(f"{j + 1:4d}: {llir_lines[j]}")


SSA Value: %117 =
 120:   %115 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 504
 121:   br label %116, !dbg !19
 122: 
 123: 116:                                              ; preds = %5, %116
 124:   %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %116 ]
 125:   %117 = phi <16 x float> [ zeroinitializer, %5 ], [ %419, %116 ]
 126:   %118 = shl nuw nsw i64 %indvars.iv, 5, !dbg !20
 127:   %119 = or disjoint i64 %118, %85, !dbg !21
 128:   %120 = or disjoint i64 %118, %86, !dbg !21
 129:   %gep318 = getelementptr half, ptr addrspace(1) %invariant.gep317, i64 %118, !dbg !22
 130:   %gep.idx = shl i64 %119, 10, !dbg !23

SSA Value: %419 =
 437:   %414 = tail call <16 x float> @llvm.fmuladd.v16f32(<16 x float> %413, <16 x float> %266, <16 x float> %409), !dbg !26
 438:   %415 = insertelement <16 x float> poison, float %153, i64 0, !dbg !26
 439:   %416 = insertelement <16 x float> %415, float %155, i64 1, !dbg !26
 440:   %417 = insertelement <16 x float> %416, float %


### Step 2.3 — Key Findings: Loop-Carried Accumulator

The LLVM IR contains a loop-carried FP32 accumulator represented by a phi node:

```llvm
%117 = phi <16 x float>
    [ zeroinitializer, %5 ], [ %419, %116 ]
```

The phi node selects a zero-initialized vector on the first iteration and the updated accumulator from the loop back edge on subsequent iterations.

The updated accumulator is generated through vector floating-point multiply-add operations, including:

```llvm
%419 = tail call <16 x float> @llvm.fmuladd.v16f32(...)
```

This demonstrates that the GEMM dot computation has been lowered into LLVM vector arithmetic operations.

The LLVM IR also contains `insertelement` and `shufflevector` instructions for constructing and rearranging vector operands.

These observations are consistent with the previously inspected PTX, which contains FP32 FMA instructions rather than Tensor Core MMA instructions.

The analysis demonstrates how GEMM accumulation is implemented using LLVM SSA, loop-carried phi nodes, and lower-level floating-point arithmetic.


### Task 4 — Reconstruct the Loop CFG

In [27]:
# ============================================================
# Step 2.4: Reconstruct LLVM Control Flow
# ============================================================

# 1: Find basic block labels
basic_blocks = [
    (i, line.strip())
    for i, line in enumerate(llir_lines)
    if re.match(
        r"^\s*(?:[A-Za-z$._][\w$.-]*|\d+):(?:\s|$)",
        line
    )
]

print("Basic Block Labels:")

for i, line in basic_blocks:
    print(f"{i + 1:4d}: {line}")


# 2: Find branch instructions
branches = [
    (i, line.strip())
    for i, line in enumerate(llir_lines)
    if re.match(r"^\s*br\s", line)
]

print("\nBranch Instructions:")

for i, line in branches:
    print(f"{i + 1:4d}: {line}")


# 3 (Manual Analysis):
#
# Identify:
#
# Entry Block:
# Loop Block:
# Exit Block:
#
# Back Edge:
# Exit Edge:
print("\n" + "=" * 70)
print("CFG Analysis")
print("=" * 70)

print("Entry Block: %5")
print("Loop Block:  %116")
print("Exit Block:  %420")

print("\nBack Edge: %116 -> %116")
print("Exit Edge: %116 -> %420")

Basic Block Labels:
 123: 116:                                              ; preds = %5, %116
 447: 420:                                              ; preds = %116

Branch Instructions:
 121: br label %116, !dbg !19
 445: br i1 %exitcond.not, label %420, label %116, !dbg !19

CFG Analysis
Entry Block: %5
Loop Block:  %116
Exit Block:  %420

Back Edge: %116 -> %116
Exit Edge: %116 -> %420


### Task 5 — Compare TTGIR and LLVM Loop Structures

In [28]:
# ============================================================
# Step 2.5: Compare TTGIR and LLVM Loop Structures
# ============================================================

# Retrieve TTGIR
ttgir = str(asm_dict["ttgir"])
ttgir_lines = ttgir.splitlines()

# 1 Find TTGIR structured loop operations
ttgir_keywords = [
    "scf.for",
    "scf.yield",
]

for keyword in ttgir_keywords:

    matches = [
        line.strip()
        for line in ttgir_lines
        if keyword in line
    ]

    print("\nTTGIR Operation:", keyword)

    for line in matches:
        print(" ", line)


# 2 Find corresponding LLVM operations
llvm_keywords = [
    " phi ",
    "br ",
]

for keyword in llvm_keywords:

    matches = [
        line.strip()
        for line in llir_lines
        if keyword in line
        and not line.lstrip().startswith(";")
    ]

    print("\nLLVM Operation:", keyword)

    for line in matches[:5]:
        print(" ", line.strip())


#3 (Manual Analysis):
#
# Explain the following transformations:
#
# scf.for      -> ?
# iter_args    -> ?
# scf.yield    -> ?
# Loop exit    -> ?
print("\n" + "=" * 70)
print("TTGIR to LLVM Loop Lowering")
print("=" * 70)

print("scf.for   -> LLVM Basic Blocks + Branches")
print("iter_args -> LLVM Phi Nodes")
print("scf.yield -> Loop-Carried SSA Values")
print("Loop exit -> Conditional Branch")


TTGIR Operation: scf.for
  %accumulator = scf.for %accumulator_32 = %c0_i32 to %c8_i32 step %c1_i32 iter_args(%accumulator_33 = %cst_6) -> (tensor<32x64xf32, #blocked2>)  : i32 {

TTGIR Operation: scf.yield
  scf.yield %accumulator_58 : tensor<32x64xf32, #blocked2> loc(#loc32)

LLVM Operation:  phi 
  %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %116 ]
  %117 = phi <16 x float> [ zeroinitializer, %5 ], [ %419, %116 ]

LLVM Operation: br 
  br label %116, !dbg !19
  br i1 %exitcond.not, label %420, label %116, !dbg !19

TTGIR to LLVM Loop Lowering
scf.for   -> LLVM Basic Blocks + Branches
iter_args -> LLVM Phi Nodes
scf.yield -> Loop-Carried SSA Values
Loop exit -> Conditional Branch



### Step 2 — Key Findings: LLVM Loop Lowering

The GEMM reduction loop is represented using structured control flow in TTGIR and explicit SSA-based control flow in LLVM IR.

**TTGIR Representation**

- `scf.for` defines the eight-iteration K-tile reduction loop.
- `iter_args` carries the FP32 accumulator across iterations.
- `scf.yield` produces the updated accumulator value.

**LLVM IR Representation**

- The generated kernel contains three basic blocks: entry, loop, and exit.
- The loop block has a self-loop back edge.
- Two phi nodes maintain the loop induction variable and FP32 accumulator.
- Integer addition and comparison instructions update the iteration count and evaluate the exit condition.
- The GEMM accumulation is implemented using vector floating-point multiply-add operations.

**Cross-Stage Mapping**

| TTGIR | LLVM IR |
|---|---|
| `scf.for` | Basic blocks and branches |
| Loop induction variable | Induction-variable phi node |
| `iter_args` | Accumulator phi node |
| `scf.yield` | Updated SSA value on the back edge |
| Loop termination | `icmp` and conditional `br` |

The LLVM IR uses an induction variable that advances from 0 to 7, corresponding to eight K-tile iterations.

This experiment demonstrates how structured loops and loop-carried dependencies are lowered into explicit LLVM SSA and CFG representations.



## Step 3 — Analyze LLVM Memory Address Spaces

### Objective

The objective of this step is to investigate how Triton lowers GPU memory operations into LLVM IR, with a focus on global memory, shared memory, address calculation, and memory access instructions.

In Experiment 1.4, we observed that the compiler introduced shared-memory-backed operations to prepare GEMM dot operands.

In this experiment, we will examine how these operations are represented in LLVM IR and how they relate to the generated PTX instructions.

### Background — GPU Memory Address Spaces

LLVM IR uses address spaces to distinguish pointers that refer to different GPU memory regions.

For the NVIDIA NVPTX target, two important address spaces are:

- `addrspace(1)` — Global Memory
- `addrspace(3)` — Shared Memory

For example:

```llvm
%ptr = getelementptr half, ptr addrspace(1) %A, i64 %offset
```

This computes the address of an FP16 element in global memory.

```llvm
%ptr = getelementptr i8, ptr addrspace(3) @global_smem, i32 %offset
```

This computes an address within shared memory.

Importantly, `getelementptr` performs address calculation. It does not load or store memory.

### Task 1 — Identify LLVM Address Spaces

Search the LLVM IR for:

- `addrspace(1)`
- `addrspace(3)`
- `@global_smem`

Identify which instructions reference global memory and which reference shared memory.

### Task 2 — Analyze GetElementPtr Instructions

Find `getelementptr` instructions and inspect their pointer types, offsets, and address spaces.

Investigate how the compiler calculates addresses for GEMM matrix tiles and shared-memory buffers.

### Task 3 — Inspect LLVM Load and Store Operations

Search for:

- `load`
- `store`
- `llvm.nvvm`
- Inline assembly memory instructions

Distinguish ordinary LLVM memory operations from target-specific inline PTX assembly.

### Task 4 — Compare LLVM IR and PTX Memory Instructions

Compare the generated LLVM IR with PTX operations such as:

- `ld.global`
- `st.global`
- `ld.shared`
- `st.shared`

Investigate how global-memory accesses and shared-memory operations are represented across compiler stages.

### Questions to Investigate

1. How does LLVM IR distinguish global memory from shared memory?
2. What is the difference between `getelementptr` and `load`?
3. Why does the compiler generate many address calculation instructions?
4. How are Triton shared-memory operations represented at lower compiler levels?
5. Why can TTGIR contain no `ttg.local_store` while PTX still contains `st.shared` instructions?

### Expected Outcome

By the end of this step, we should be able to:

1. Explain LLVM GPU memory address spaces.
2. Identify global and shared memory pointers in LLVM IR.
3. Understand the role of `getelementptr` in GPU address calculation.
4. Recognize LLVM memory operations and inline PTX assembly.
5. Connect TTGIR memory operations to generated PTX memory instructions.




### Task 3.1 — Identify LLVM Address Spaces

In [29]:
# ============================================================
# Experiment 1.5 — LLVM IR & PTX Analysis
# Step 3.1: Identify LLVM Address Spaces
# ============================================================

address_keywords = [
    "addrspace(1)",
    "addrspace(3)",
    "@global_smem",
]

for keyword in address_keywords:

    # 1: Find matching lines
    matches = [
        (i, line)
        for i, line in enumerate(llir_lines)
        if keyword in line
    ]

    print("\n" + "=" * 70)
    print("Keyword:", keyword)
    print("=" * 70)

    print("Matches:", len(matches))

    # 2: Print the first 10 matches
    for i, line in matches[:10]:
        print(f"{i + 1:4d}: {line.strip()}")


Keyword: addrspace(1)
Matches: 16
   8: define ptx_kernel void @gemm_kernel(ptr addrspace(1) %0, ptr addrspace(1) %1, ptr addrspace(1) %2, ptr addrspace(1) readnone captures(none) %3, ptr addrspace(1) readnone captures(none) %4) local_unnamed_addr #0 !dbg !4 {
  27: %24 = getelementptr half, ptr addrspace(1) %0, i64 %23, !dbg !16
  31: %invariant.gep = getelementptr half, ptr addrspace(1) %1, i64 %27, !dbg !19
  91: %invariant.gep317 = getelementptr half, ptr addrspace(1) %24, i64 %84, !dbg !19
 129: %gep318 = getelementptr half, ptr addrspace(1) %invariant.gep317, i64 %118, !dbg !22
 131: %gep = getelementptr i8, ptr addrspace(1) %invariant.gep, i64 %gep.idx, !dbg !23
 133: %gep286 = getelementptr i8, ptr addrspace(1) %invariant.gep, i64 %gep286.idx, !dbg !23
 134: %121 = tail call { i32, i32, i32, i32 } asm sideeffect "mov.u32 $0, $4;\0A\09mov.u32 $1, $5;\0A\09mov.u32 $2, $6;\0A\09mov.u32 $3, $7;\0A\09@$9 ld.global.v4.b32 { $0, $1, $2, $3 }, [ $8 + 0 ];", "=r,=r,=r,=r,r,r,r,r,l,b"(i


### Step 3.1 — Key Findings: LLVM Address Spaces

The LLVM IR analysis identified 16 lines referencing global memory address space 1 and 183 lines referencing shared memory address space 3.

The GEMM kernel parameters for matrices A, B, and C are represented as global-memory pointers using `ptr addrspace(1)`.

Shared memory is represented using the `@global_smem` symbol in address space 3.

The compiler generates `getelementptr` instructions to calculate addresses for both global and shared memory.

For global memory, address calculations are based on matrix offsets and GEMM tile indices.

For shared memory, numerous GEP instructions calculate offsets relative to the shared-memory base pointer or previously computed pointers.

The LLVM IR also contains inline PTX assembly for vectorized, predicated global-memory loads, including `ld.global.v4.b32`.

These observations show how Triton's high-level tensor memory operations are lowered into explicit pointer arithmetic and target-specific GPU memory instructions.

The reported keyword counts reflect LLVM IR text matches rather than dynamic memory access counts.


In [30]:
# ============================================================
# Step 3.2: Analyze LLVM GetElementPtr Instructions
# ============================================================

# 1. Find all getelementptr instructions
gep_lines = [
    line.strip()
    for line in llir_lines
    if "getelementptr" in line
]

print("Total GEP Instructions:", len(gep_lines))

# 2. Separate global and shared memory GEPs
global_geps = [
    line for line in gep_lines
    if "addrspace(1)" in line
]

shared_geps = [
    line for line in gep_lines
    if "addrspace(3)" in line
]

print("Global Memory GEPs:", len(global_geps))
print("Shared Memory GEPs:", len(shared_geps))

print("\nGlobal Memory Examples:")
for line in global_geps[:5]:
    print(" ", line)

print("\nShared Memory Examples:")
for line in shared_geps[:5]:
    print(" ", line)

Total GEP Instructions: 100
Global Memory GEPs: 10
Shared Memory GEPs: 90

Global Memory Examples:
  %24 = getelementptr half, ptr addrspace(1) %0, i64 %23, !dbg !16
  %invariant.gep = getelementptr half, ptr addrspace(1) %1, i64 %27, !dbg !19
  %invariant.gep317 = getelementptr half, ptr addrspace(1) %24, i64 %84, !dbg !19
  %gep318 = getelementptr half, ptr addrspace(1) %invariant.gep317, i64 %118, !dbg !22
  %gep = getelementptr i8, ptr addrspace(1) %invariant.gep, i64 %gep.idx, !dbg !23

Shared Memory Examples:
  %30 = getelementptr inbounds nuw i8, ptr addrspace(3) @global_smem, i32 %29
  %31 = getelementptr inbounds nuw i8, ptr addrspace(3) %30, i32 16
  %33 = getelementptr inbounds nuw i8, ptr addrspace(3) @global_smem, i32 %32
  %34 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 16
  %35 = getelementptr inbounds nuw i8, ptr addrspace(3) %33, i32 32



### Step 3.2 — Key Findings: LLVM GetElementPtr Analysis

The generated LLVM IR contains 100 `getelementptr` instructions:

- 10 global-memory GEP instructions using `addrspace(1)`.
- 90 shared-memory GEP instructions using `addrspace(3)`.

Global-memory GEP instructions calculate addresses for the GEMM input and output matrices.

For example, the compiler uses the loop induction variable to calculate K-tile offsets and derive global-memory pointers.

Shared-memory GEP instructions calculate offsets relative to `@global_smem` or previously generated shared-memory pointers.

The large number of shared-memory GEP instructions is consistent with the shared-memory-backed dot operand preparation observed in TTGIR.

Importantly, GEP instructions perform pointer arithmetic without accessing memory. Their counts do not represent the number of runtime memory accesses or memory transactions.

LLVM and GPU backend optimizations may also simplify address calculations or incorporate offsets into target-specific memory instructions.

This experiment demonstrates how high-level GPU tensor layouts and memory operations are lowered into explicit pointer computations.


In [31]:
# ============================================================
# Optional — Inspect GEP Context
# ============================================================

for i, line in enumerate(llir_lines):

    if (
        "getelementptr" in line
        and "addrspace(1)" in line
    ):

        start = max(0, i - 3)
        end = min(len(llir_lines), i + 4)

        print("\n" + "=" * 70)
        print(f"Global GEP at Line {i + 1}")
        print("=" * 70)

        for j in range(start, end):
            print(f"{j + 1:4d}: {llir_lines[j]}")

        # Limit output to the first example
        break


Global GEP at Line 27
  24:   %21 = or disjoint i32 %20, %18, !dbg !14
  25:   %22 = shl i32 %17, 8, !dbg !15
  26:   %23 = sext i32 %22 to i64, !dbg !16
  27:   %24 = getelementptr half, ptr addrspace(1) %0, i64 %23, !dbg !16
  28:   %25 = icmp slt i32 %17, 256, !dbg !17
  29:   %26 = icmp samesign ult i32 %7, 8, !dbg !18
  30:   %27 = zext nneg i32 %21 to i64



### Optional Analysis — Global Memory GEP Context

The LLVM IR reveals how the compiler calculates global-memory addresses for the GEMM input matrix.

The instruction `shl i32 %17, 8` multiplies a row-related index by 256, which corresponds to the row stride of matrix A.

The result is sign-extended to 64 bits and used by a `getelementptr half` instruction to compute an address in global memory.

Since each FP16 element occupies two bytes, the GEP converts an element offset into the corresponding byte address.

The LLVM IR also includes integer comparison instructions for boundary checking, consistent with the masked memory operations in the original Triton kernel.

These observations demonstrate how high-level tensor indexing is lowered into integer arithmetic, pointer calculations, and boundary predicates.


In [32]:
# ============================================================
# Step 3.3: Inspect LLVM Memory Instructions
# ============================================================

memory_keywords = [
    "load ",
    "store ",
    "asm ",
    "llvm.nvvm",
]

for keyword in memory_keywords:

    # 1. Find matching lines
    matches = [
        line for line in llir_lines
        if keyword in line
        and not line.lstrip().startswith(";")
    ]

    print("\n" + "=" * 70)
    print("Keyword:", keyword)
    print("=" * 70)

    print("Matches:", len(matches))

    # 2. Print the first 8 matches
    for line in matches[:8]:
        print(" ", line.strip())


Keyword: load 
Matches: 82
  %152 = load float, ptr addrspace(3) %36, align 16, !dbg !26
  %153 = load float, ptr addrspace(3) %93, align 4, !dbg !26
  %154 = load float, ptr addrspace(3) %40, align 16, !dbg !26
  %155 = load float, ptr addrspace(3) %100, align 4, !dbg !26
  %156 = load float, ptr addrspace(3) %44, align 16, !dbg !26
  %157 = load float, ptr addrspace(3) %107, align 4, !dbg !26
  %158 = load float, ptr addrspace(3) %110, align 8, !dbg !26
  %159 = load float, ptr addrspace(3) %114, align 4, !dbg !26

Keyword: store 
Matches: 10
  store <4 x float> %149, ptr addrspace(3) %30, align 16, !dbg !26
  store <4 x float> %151, ptr addrspace(3) %31, align 16, !dbg !26
  store <4 x float> %161, ptr addrspace(3) %30, align 16, !dbg !26
  store <4 x float> %163, ptr addrspace(3) %31, align 16, !dbg !26
  store <4 x float> %165, ptr addrspace(3) %48, align 16, !dbg !26
  store <4 x float> %167, ptr addrspace(3) %49, align 16, !dbg !26
  store <4 x half> %473, ptr addrspace(3) %469


### Step 3.3 — Key Findings: LLVM Memory Instructions

The generated LLVM IR contains both native LLVM memory instructions and target-specific inline PTX assembly.

**Shared Memory Operations**

The keyword search identified 82 lines containing `load` and 10 lines containing `store`.

The inspected native LLVM memory instructions use address space 3, which represents GPU shared memory.

Examples include scalar FP32 loads and vectorized FP32/FP16 stores.

**Global Memory Operations**

Five inline assembly statements were identified:

- Three vectorized global-memory loads using `ld.global.v4.b32`.
- Two vectorized global-memory stores using `st.global.v4.b32`.

These operations use inline PTX assembly to express target-specific vectorized and predicated memory accesses.

**NVVM Intrinsics**

The LLVM IR also contains NVVM intrinsics for CTA indexing, thread indexing, and CTA-level synchronization.

Examples include `llvm.nvvm.read.ptx.sreg.ctaid.x`, `llvm.nvvm.read.ptx.sreg.tid.x`, and `llvm.nvvm.barrier.cta.sync.aligned.all`.

**Conclusion**

This experiment demonstrates that Triton GPU memory lowering can use both native LLVM memory instructions and inline PTX assembly.

The compiler represents shared-memory operations with LLVM loads and stores, while the inspected global-memory accesses use target-specific inline assembly.

The resulting PTX contains explicit global-memory and shared-memory instructions.


In [33]:
# ============================================================
# Optional — Classify LLVM Memory Instructions
# ============================================================

import re

instruction_patterns = {
    "LLVM Load": r"^\s*(?:%[\w.$-]+\s*=\s*)?load\s",
    "LLVM Store": r"^\s*store\s",
    "Inline Assembly": r"\basm\b",
    "NVVM Intrinsic": r"@llvm\.nvvm\.",
}

for name, pattern in instruction_patterns.items():

    matches = [
        line.strip()
        for line in llir_lines
        if re.search(pattern, line)
        and not line.lstrip().startswith(";")
    ]

    print(f"\n{name}: {len(matches)} matches")

    for line in matches[:5]:
        print(" ", line)


LLVM Load: 82 matches
  %152 = load float, ptr addrspace(3) %36, align 16, !dbg !26
  %153 = load float, ptr addrspace(3) %93, align 4, !dbg !26
  %154 = load float, ptr addrspace(3) %40, align 16, !dbg !26
  %155 = load float, ptr addrspace(3) %100, align 4, !dbg !26
  %156 = load float, ptr addrspace(3) %44, align 16, !dbg !26

LLVM Store: 10 matches
  store <4 x float> %149, ptr addrspace(3) %30, align 16, !dbg !26
  store <4 x float> %151, ptr addrspace(3) %31, align 16, !dbg !26
  store <4 x float> %161, ptr addrspace(3) %30, align 16, !dbg !26
  store <4 x float> %163, ptr addrspace(3) %31, align 16, !dbg !26
  store <4 x float> %165, ptr addrspace(3) %48, align 16, !dbg !26

Inline Assembly: 5 matches
  %121 = tail call { i32, i32, i32, i32 } asm sideeffect "mov.u32 $0, $4;\0A\09mov.u32 $1, $5;\0A\09mov.u32 $2, $6;\0A\09mov.u32 $3, $7;\0A\09@$9 ld.global.v4.b32 { $0, $1, $2, $3 }, [ $8 + 0 ];", "=r,=r,=r,=r,r,r,r,r,l,b"(i32 0, i32 0, i32 0, i32 0, ptr addrspace(1) %gep318, i1 %

In [34]:
# ============================================================
# Experiment 1.5 — LLVM IR & PTX Analysis
# Step 3.4: Compare LLVM IR and PTX Memory Operations
# ============================================================

ptx_memory_keywords = [
    "ld.global",
    "st.global",
    "ld.shared",
    "st.shared",
]

for keyword in ptx_memory_keywords:

    # 1. Find matching PTX instructions
    matches = [
        line.strip()
        for line in ptx_lines
        if keyword in line
        and not line.lstrip().startswith("//")
    ]

    print("\n" + "=" * 70)
    print("PTX Instruction:", keyword)
    print("=" * 70)

    print("Matches:", len(matches))

    # 2. Print the first 5 instructions
    for line in matches[:5]:
        print(" ", line)


PTX Instruction: ld.global
Matches: 3
  @%p1 ld.global.v4.b32 { %r27, %r28, %r29, %r30 }, [ %rd11 + 0 ];
  @%p2 ld.global.v4.b32 { %r32, %r33, %r34, %r35 }, [ %rd18 + 0 ];
  @%p2 ld.global.v4.b32 { %r36, %r37, %r38, %r39 }, [ %rd12 + 0 ];

PTX Instruction: st.global
Matches: 2
  @%p4 st.global.v4.b32 [ %rd13 + 0 ], { %r816, %r817, %r818, %r819 };
  @%p4 st.global.v4.b32 [ %rd14 + 0 ], { %r820, %r821, %r822, %r823 };

PTX Instruction: ld.shared
Matches: 116
  ld.shared.b32 	%r48, [%r10+80];
  ld.shared.b32 	%r49, [%r10+124];
  ld.shared.b32 	%r50, [%r10+208];
  ld.shared.b32 	%r51, [%r10+252];
  ld.shared.b32 	%r52, [%r10+336];

PTX Instruction: st.shared
Matches: 10
  st.shared.v4.b32 	[%r9], {%r43, %r42, %r41, %r40};
  st.shared.v4.b32 	[%r9+16], {%r47, %r46, %r45, %r44};
  st.shared.v4.b32 	[%r9], {%r59, %r58, %r57, %r56};
  st.shared.v4.b32 	[%r9+16], {%r63, %r62, %r61, %r60};
  st.shared.v4.b32 	[%r9+4096], {%r67, %r66, %r65, %r64};



### Step 3.4 — Key Findings: LLVM IR and PTX Memory Operations

The PTX analysis identified the following static memory instruction counts:

- 3 global-memory loads using `ld.global.v4.b32`.
- 2 global-memory stores using `st.global.v4.b32`.
- 116 shared-memory load instructions.
- 10 shared-memory store instructions.

**Global Memory Operations**

The generated PTX uses 128-bit vectorized and predicated global-memory loads and stores.

The predicates are consistent with the masked memory operations in the original Triton GEMM kernel.

**Shared Memory Operations**

The generated PTX contains scalar shared-memory loads and vectorized shared-memory stores.

These instructions are consistent with the shared-memory-backed dot operand preparation observed in TTGIR.

**Cross-Stage Comparison**

The LLVM IR contains native shared-memory load/store operations and inline PTX assembly for global-memory accesses.

The inspected inline assembly corresponds to the global-memory instructions in the generated PTX.

The number of LLVM shared-memory operations does not necessarily match the number of PTX instructions because backend lowering may split, combine, or transform memory operations.

**Conclusion**

This experiment demonstrates how Triton lowers high-level GPU memory operations into LLVM IR and target-specific PTX instructions.

The results also illustrate the differences between address calculation, memory access representation, instruction vectorization, and runtime memory traffic.
